# Data Preprocessing
This notebook converts relational sales and operational data into a clean, machine-learning-ready dataset for a **one-month-ahead product risk model**.

## Business objective and unit of analysis

The model should answer:

> Based on all information available at the end of the current month, which products are likely to show medium or high commercial risk next month?

| Item | Definition |
|---|---|
| Observation grain | One row per `product_id × region × year_month` |
| Prediction point | End of the current month |
| Prediction horizon | One month ahead |
| Target | `next_month_risk_label`: 0 = low, 1 = medium, 2 = high |
| Split strategy | Chronological 70% train / 15% validation / 15% test |

This timing assumption matters. Current-month sales, CRM, pipeline, inventory and market information may be used only because the prediction is assumed to run **after the month closes**.

## 1. Setup

In [1]:
from pathlib import Path
import json
import csv

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

RANDOM_SEED = 42
TARGET_COL = "next_month_risk_label"
ID_COLS = ["product_id", "year_month", "region"]

INPUT_DIR = Path("../data/full_data/sql_input")
OUTPUT_DIR_PREPROCESSED = Path("../data/full_data/preprocessed_data")

OUTPUT_DIR_PREPROCESSED.mkdir(parents=True, exist_ok=True)

print(f"Input folder : {INPUT_DIR.resolve()}")
print(f"Output folder: {OUTPUT_DIR_PREPROCESSED.resolve()}")


Input folder : C:\Benutzer\Anastasia\Lokale Daten\AS Portfolio\early-warning\data\full_data\sql_input
Output folder: C:\Benutzer\Anastasia\Lokale Daten\AS Portfolio\early-warning\data\full_data\preprocessed_data


## 2. Load and inspect the source tables

In [2]:
CSV_FILES = {
    "sales": "sales.csv",
    "products": "products.csv",
    "customers": "customers.csv",
    "regions": "regions.csv",
    "sales_reps": "sales_reps.csv",
    "inventory": "inventory.csv",
    "costs": "costs.csv",
    "returns": "returns.csv",
    "crm": "crm_activities.csv",
    "pipeline": "pipeline.csv",
    "date": "dates.csv",
    "market_activity": "market_activity.csv",
    "market_signals": "market_signals.csv",
}

def detect_csv_settings(path: Path) -> tuple[str, str]:
    """Return field separator and decimal marker for a CSV file."""
    sample = path.read_text(encoding="utf-8-sig", errors="replace")[:8192]
    try:
        separator = csv.Sniffer().sniff(
            sample, delimiters=",;\t|"
        ).delimiter
    except csv.Error:
        separator = ";"
    decimal_marker = "," if separator == ";" else "."
    return separator, decimal_marker

def load_csv_tables(
    input_dir: Path, file_map: dict[str, str]
) -> tuple[dict[str, pd.DataFrame], pd.DataFrame]:
    """Load every required source and return the data plus load metadata."""
    missing_files = [
        filename
        for filename in file_map.values()
        if not (input_dir / filename).exists()
    ]
    if missing_files:
        raise FileNotFoundError(
            f"Missing files in {input_dir}: {missing_files}"
        )

    datasets = {}
    load_rows = []

    for table_name, filename in file_map.items():
        path = input_dir / filename
        separator, decimal_marker = detect_csv_settings(path)
        dataframe = pd.read_csv(
            path,
            sep=separator,
            decimal=decimal_marker,
            low_memory=False,
        )
        datasets[table_name] = dataframe
        load_rows.append(
            {
                "table": table_name,
                "file": filename,
                "separator": repr(separator),
                "decimal": decimal_marker,
                "rows": len(dataframe),
                "columns": dataframe.shape[1],
            }
        )

    return datasets, pd.DataFrame(load_rows)

raw, load_summary = load_csv_tables(INPUT_DIR, CSV_FILES)

print(f"Loaded {len(raw)} source tables.")
display(load_summary)


Loaded 13 source tables.


,table,file,separator,decimal,rows,columns
0,sales,sales.csv,"','",.,120000,26
1,products,products.csv,"','",.,200,14
2,customers,customers.csv,"','",.,1200,9
3,regions,regions.csv,"','",.,10,7
4,sales_reps,sales_reps.csv,"','",.,85,5
5,inventory,inventory.csv,"','",.,69852,10
6,costs,costs.csv,"','",.,12000,7
7,returns,returns.csv,"','",.,4200,10
8,crm,crm_activities.csv,"','",.,35000,10
9,pipeline,pipeline.csv,"','",.,15000,14


## 2.1 High-level data profile

Before preprocessing, we record row counts, widths, duplicate rows, missingness, and memory use. Unexpected changes here often reveal an incorrect export before they become a difficult downstream error.

In [3]:
raw_overview = pd.DataFrame(
    [
        {
            "table": table_name,
            "rows": len(dataframe),
            "columns": dataframe.shape[1],
            "duplicate_rows": int(dataframe.duplicated().sum()),
            "missing_cells_pct": round(
                dataframe.isna().mean().mean() * 100, 2
            ),
            "memory_mb": round(
                dataframe.memory_usage(deep=True).sum() / 1_048_576,
                2,
            ),
        }
        for table_name, dataframe in raw.items()
    ]
).sort_values("table", ignore_index=True)

display(raw_overview)


,table,rows,columns,duplicate_rows,missing_cells_pct,memory_mb
0,costs,12000,7,0,0.000,1.220
1,crm,35000,10,0,0.000,7.520
2,customers,1200,9,0,0.000,0.380
3,date,4018,18,0,0.000,1.990
4,inventory,69852,10,0,0.000,8.260
5,market_activity,120000,16,0,5.110,26.680
6,market_signals,6000,14,0,0.000,1.540
7,pipeline,15000,14,0,0.000,4.580
8,products,200,14,0,0.000,0.080
9,regions,10,7,0,0.000,0.000


In [4]:
# View small examples without printing entire source tables.
for table_name in [
    "sales",
    "products",
    "customers",
    "inventory",
    "market_activity",
    "market_signals",
]:
    print(f"\n{table_name.upper()} — first 2 rows")
    display(raw[table_name].head(2))



SALES — first 2 rows


,sales_id,order_id,date_id,order_date,year_month,customer_id,product_id,region_id,sales_rep_id,units,asp_eur,discount_pct,revenue_eur,revenue_local_currency,currency,unit_cost_eur,gross_profit_eur,gross_margin_pct,is_outlier_order,cogs_eur,discount_value_eur,margin_category,order_size_category,missing_sales_rep_flag,missing_discount_flag,missing_margin_flag
0,1,ORD-00000001,20220304,2022-03-04,2022-03-01,282,1133,2,39.000,26,104.000,0.092,"2,704.130","2,704.130",EUR,78.470,663.790,0.245,False,"2,040.340",248.780,Low Margin,Medium Order,0,0,0
1,2,ORD-00000002,20240503,2024-05-03,2024-05-01,562,1105,2,73.000,21,"1,698.070",0.089,"35,659.450","35,659.450",EUR,944.570,"15,823.420",0.444,False,"19,836.030","3,173.691",High Margin,Small Order,0,0,0



PRODUCTS — first 2 rows


,product_id,sku,product_family,product_group,product_name,launch_year,lifecycle_stage,base_list_price_eur,base_unit_cost_eur,target_margin_pct,product_growth_factor,is_declining_product,is_new_product,launch_date
0,1000,SKU-1000,IT Devices,Laptop Pro,Laptop Pro Model 01,2022,Decline,804.370,540.920,0.290,1.080,1,0,2022-01-01
1,1001,SKU-1001,IT Devices,Laptop Pro,Laptop Pro Model 02,2023,Growth,771.330,542.710,0.290,1.080,0,0,2023-01-01



CUSTOMERS — first 2 rows


...

,inventory_id,year_month,product_id,region_id,opening_stock_units,production_units,ending_stock_units,stockout_flag,inventory_value_eur,zero_stock_flag
0,1,2021-01-01,1000,1,53,79,51,False,"30,108.390",0
1,2,2021-01-01,1000,3,31,22,34,False,"11,838.230",0



MARKET_ACTIVITY — first 2 rows


,year_month,product_id,region_id,campaign_flag,campaign_channel,campaign_spend_eur,campaign_impressions,campaign_clicks,campaign_ctr_pct,website_visits,product_page_views,demo_requests,marketing_qualified_leads,cost_per_lead_eur,regional_crm_activity_index,pipeline_interest_index
0,2021-01,1000,1,0,No Active Campaign,0.000,0,0,0.000,270,514,1,1,NaN,194.890,119.520
1,2021-02,1000,1,0,No Active Campaign,0.000,0,0,0.000,264,354,2,2,NaN,205.660,179.370



MARKET_SIGNALS — first 2 rows


,year_month,product_family,product_group,region_id,market_demand_index,market_growth_pct,competitor_pressure_index,seasonality_index,macro_business_index,supply_pressure_index,pipeline_interest_index,demand_shock_flag,market_opportunity_score,regional_market_growth_factor
0,2021-01,Accessories,Accessory Kit,1,102.070,0.000,71.060,100.000,100.260,41.440,278.880,0,78.980,1.060
1,2021-02,Accessories,Accessory Kit,1,98.970,-3.040,61.980,100.000,103.460,41.730,134.530,0,79.690,1.060


## 3. Validate the source schema and relational keys

A professional pipeline should fail early when an upstream table changes. The validation below lists every field that is genuinely required by this notebook.

Fields that are merely useful enrichment attributes remain optional and are selected only when present.


In [5]:
REQUIRED_SOURCE_COLUMNS = {
    "sales": [
        "sales_id", "order_date", "customer_id", "product_id",
        "region_id", "sales_rep_id", "units", "asp_eur",
        "discount_pct", "revenue_eur", "gross_profit_eur",
        "gross_margin_pct",
    ],
    "products": [
        "product_id", "product_family", "product_group",
        "launch_date", "lifecycle_stage",
    ],
    "customers": [
        "customer_id", "region_id", "customer_segment",
        "customer_size_score", "base_churn_probability",
    ],
    "regions": ["region_id", "region", "country"],
    "sales_reps": [
        "sales_rep_id", "region_id", "seniority", "annual_quota_eur"
    ],
    "inventory": [
        "year_month", "product_id", "region_id",
        "opening_stock_units", "production_units",
        "ending_stock_units", "stockout_flag", "zero_stock_flag",
        "inventory_value_eur",
    ],
    "costs": [
        "year_month", "product_id", "standard_unit_cost_eur",
        "actual_unit_cost_eur", "cost_variance_eur",
        "cost_variance_pct",
    ],
    "returns": [
        "return_id", "sales_id", "return_date", "product_id",
        "region_id", "return_units", "return_value_eur",
    ],
    "crm": [
        "activity_id", "activity_date", "customer_id",
        "sales_rep_id", "activity_type", "activity_minutes",
        "sentiment_score", "customer_health_score",
    ],
    "pipeline": [
        "opportunity_id", "created_date", "customer_id",
        "product_group", "sales_rep_id", "expected_value_eur",
        "win_probability", "weighted_pipeline_eur",
        "is_closed_won", "is_closed_lost",
    ],
    "date": ["Date", "Year", "Quarter", "Month", "MonthName"],
    "market_activity": [
        "year_month", "product_id", "region_id", "campaign_flag",
        "campaign_channel", "campaign_spend_eur",
        "campaign_impressions", "campaign_clicks",
        "campaign_ctr_pct", "website_visits", "product_page_views",
        "demo_requests", "marketing_qualified_leads",
        "cost_per_lead_eur", "regional_crm_activity_index",
        "pipeline_interest_index",
    ],
    "market_signals": [
        "year_month", "product_family", "product_group", "region_id",
        "market_demand_index", "market_growth_pct",
        "competitor_pressure_index", "seasonality_index",
        "macro_business_index", "supply_pressure_index",
        "pipeline_interest_index", "demand_shock_flag",
        "market_opportunity_score", "regional_market_growth_factor",
    ],
}

schema_issues = []
for table_name, required_columns in REQUIRED_SOURCE_COLUMNS.items():
    if table_name not in raw:
        schema_issues.append(
            {"table": table_name, "missing_column": "<table missing>"}
        )
        continue
    for column in required_columns:
        if column not in raw[table_name].columns:
            schema_issues.append(
                {"table": table_name, "missing_column": column}
            )

schema_issues = pd.DataFrame(
    schema_issues, columns=["table", "missing_column"]
)

if not schema_issues.empty:
    display(schema_issues)
    raise ValueError("Source schema validation failed.")

print("Schema validation passed for all 13 source tables.")


Schema validation passed for all 13 source tables.


In [6]:
# These are the natural keys expected before aggregation.
source_grains = {
    "products": ["product_id"],
    "customers": ["customer_id"],
    "regions": ["region_id"],
    "sales_reps": ["sales_rep_id"],
    "sales": ["sales_id"],
    "inventory": ["year_month", "product_id", "region_id"],
    "costs": ["year_month", "product_id"],
    "market_activity": ["year_month", "product_id", "region_id"],
    "market_signals": [
        "year_month", "product_family", "product_group", "region_id"
    ],
}

grain_results = []
for table_name, key_columns in source_grains.items():
    duplicate_keys = int(raw[table_name].duplicated(key_columns).sum())
    grain_results.append(
        {
            "table": table_name,
            "grain": " × ".join(key_columns),
            "duplicate_keys": duplicate_keys,
            "passed": duplicate_keys == 0,
        }
    )

grain_results = pd.DataFrame(grain_results)
display(grain_results)

if not grain_results["passed"].all():
    raise ValueError("At least one source table violates its expected grain.")


,table,grain,duplicate_keys,passed
0,products,product_id,0,True
1,customers,customer_id,0,True
2,regions,region_id,0,True
3,sales_reps,sales_rep_id,0,True
4,sales,sales_id,0,True
5,inventory,year_month × product_id × region_id,0,True
6,costs,year_month × product_id,0,True
7,market_activity,year_month × product_id × region_id,0,True
8,market_signals,year_month × product_family × product_group × ...,0,True


## 4. Standardize dimension tables

In [7]:
def normalize_id(series: pd.Series) -> pd.Series:
    return (
        series.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .replace(
            {
                "<NA>": pd.NA,
                "nan": pd.NA,
                "None": pd.NA,
                "": pd.NA,
            }
        )
    )

def to_month(series: pd.Series) -> pd.Series:
    return (
        pd.to_datetime(series, errors="coerce", format="mixed")
        .dt.to_period("M")
        .dt.to_timestamp()
    )

def numeric(
    dataframe: pd.DataFrame, column: str, default: float = np.nan
) -> pd.Series:
    if column not in dataframe.columns:
        return pd.Series(default, index=dataframe.index, dtype="float64")
    return pd.to_numeric(dataframe[column], errors="coerce")

def existing(dataframe: pd.DataFrame, columns: list[str]) -> list[str]:
    return [column for column in columns if column in dataframe.columns]

def most_common(series: pd.Series, default: str = "Unknown") -> str:
    non_missing = series.dropna().astype(str)
    if non_missing.empty:
        return default
    mode = non_missing.mode()
    return mode.iloc[0] if not mode.empty else default


# 5. Standardize dimension tables

Dimension keys are normalized before any fact-table join. The original source columns are retained, while `product_line` and `product_category` provide stable names used by the existing EWS output schema.

In [8]:
products = raw["products"].copy()
products["product_id"] = normalize_id(products["product_id"])
products["product_line"] = products["product_family"].fillna("Unknown")
products["product_category"] = products["product_group"].fillna("Unknown")
products["launch_date"] = pd.to_datetime(
    products["launch_date"], errors="coerce"
)
products = products.drop_duplicates("product_id").reset_index(drop=True)

regions = raw["regions"].copy()
regions["region_id"] = normalize_id(regions["region_id"])
regions = regions.drop_duplicates("region_id").reset_index(drop=True)

customers = raw["customers"].copy()
customers["customer_id"] = normalize_id(customers["customer_id"])
customers["region_id"] = normalize_id(customers["region_id"])
customers["customer_since"] = pd.to_datetime(
    customers["customer_since"], errors="coerce"
)
customers = customers.drop_duplicates("customer_id").reset_index(drop=True)

sales_reps = raw["sales_reps"].copy()
sales_reps["sales_rep_id"] = normalize_id(sales_reps["sales_rep_id"])
sales_reps["region_id"] = normalize_id(sales_reps["region_id"])
sales_reps = sales_reps.drop_duplicates("sales_rep_id").reset_index(
    drop=True
)

product_dimension_columns = existing(
    products,
    [
        "product_id", "product_line", "product_category",
        "product_name", "sku", "lifecycle_stage", "launch_date",
        "is_declining_product", "is_new_product",
        "base_list_price_eur", "base_unit_cost_eur",
        "target_margin_pct", "product_growth_factor",
    ],
)
region_dimension_columns = existing(
    regions,
    [
        "region_id", "region", "country", "currency", "fx_to_eur",
        "market_growth_factor", "margin_factor",
    ],
)


In [9]:
dimension_check = pd.DataFrame(
    [
        {
            "dimension": name,
            "rows": len(dataframe),
            "unique_keys": dataframe[key].nunique(dropna=True),
            "missing_keys": int(dataframe[key].isna().sum()),
            "key_is_unique": dataframe[key].is_unique,
        }
        for name, dataframe, key in [
            ("products", products, "product_id"),
            ("customers", customers, "customer_id"),
            ("regions", regions, "region_id"),
            ("sales_reps", sales_reps, "sales_rep_id"),
        ]
    ]
)
display(dimension_check)
display(products.head(3))

assert dimension_check["missing_keys"].eq(0).all()
assert dimension_check["key_is_unique"].all()
print("Dimension-key checks passed.")


,dimension,rows,unique_keys,missing_keys,key_is_unique
0,products,200,200,0,True
1,customers,1200,1200,0,True
2,regions,10,10,0,True
3,sales_reps,85,85,0,True


,product_id,sku,product_family,product_group,product_name,launch_year,lifecycle_stage,base_list_price_eur,base_unit_cost_eur,target_margin_pct,product_growth_factor,is_declining_product,is_new_product,launch_date,product_line,product_category
0,1000,SKU-1000,IT Devices,Laptop Pro,Laptop Pro Model 01,2022,Decline,804.370,540.920,0.290,1.080,1,0,2022-01-01,IT Devices,Laptop Pro
1,1001,SKU-1001,IT Devices,Laptop Pro,Laptop Pro Model 02,2023,Growth,771.330,542.710,0.290,1.080,0,0,2023-01-01,IT Devices,Laptop Pro
2,1002,SKU-1002,IT Devices,Laptop Pro,Laptop Pro Model 03,2024,Growth,828.800,552.000,0.290,1.080,0,0,2024-01-01,IT Devices,Laptop Pro


Dimension-key checks passed.


# 6. Create the canonical sales transaction table

We keep transaction-level detail but use one consistent naming convention:

- `units` becomes `units_sold`;
- `revenue_eur` becomes `revenue`;
- dates become month-start timestamps in `year_month`;
- product, region, customer, and representative attributes come from their canonical dimensions.

Invalid key/date rows are separated into `rejected_sales` rather than silently disappearing.

In [10]:
sales = raw["sales"].copy()

for column in ["product_id", "customer_id", "region_id", "sales_rep_id"]:
    sales[column] = normalize_id(sales[column])

sales["year_month"] = to_month(
    sales["year_month"] if "year_month" in sales else sales["order_date"]
)
sales["order_date"] = pd.to_datetime(sales["order_date"], errors="coerce")
sales["units_sold"] = numeric(sales, "units")
sales["revenue"] = numeric(sales, "revenue_eur")
sales["avg_discount_pct"] = numeric(sales, "discount_pct")
sales["transaction_id"] = normalize_id(sales["sales_id"])

# Remove descriptive fields that will be reapplied from dimensions.
descriptive_columns = [
    "region", "country", "currency", "fx_to_eur",
    "market_growth_factor", "margin_factor", "product_line",
    "product_category", "product_name", "sku", "lifecycle_stage",
    "launch_date", "seniority", "annual_quota_eur",
]
sales = sales.drop(columns=descriptive_columns, errors="ignore")

sales = sales.merge(
    regions[region_dimension_columns],
    on="region_id",
    how="left",
    validate="m:1",
)
sales = sales.merge(
    products[product_dimension_columns],
    on="product_id",
    how="left",
    validate="m:1",
)

customer_attributes = existing(
    customers,
    [
        "customer_id", "customer_segment", "industry",
        "customer_since", "customer_size_score",
        "base_churn_probability", "churn_risk_band",
    ],
)
sales = sales.merge(
    customers[customer_attributes],
    on="customer_id",
    how="left",
    validate="m:1",
)

representative_attributes = existing(
    sales_reps,
    ["sales_rep_id", "seniority", "annual_quota_eur"],
)
sales = sales.merge(
    sales_reps[representative_attributes],
    on="sales_rep_id",
    how="left",
    validate="m:1",
)

sales["region"] = sales["region"].fillna("Unknown")
sales["product_line"] = sales["product_line"].fillna("Unknown")
sales["product_category"] = sales["product_category"].fillna("Unknown")
sales["customer_segment"] = sales["customer_segment"].fillna("Unknown")
sales["churn_risk_band"] = sales["churn_risk_band"].fillna("Unknown")
sales["seniority"] = sales["seniority"].fillna("Unknown")

invalid_sales_rows = (
    sales["year_month"].isna()
    | sales["product_id"].isna()
    | sales["region_id"].isna()
    | sales["transaction_id"].isna()
    | sales["units_sold"].isna()
    | sales["revenue"].isna()
)
rejected_sales = sales.loc[invalid_sales_rows].copy()
sales = sales.loc[~invalid_sales_rows].reset_index(drop=True)

# Preserve configured dates, but derive effective dates from observed history.
# This prevents valid transactions from being removed when a source dimension
# contains a later launch/customer-start date than the first observed sale.
observed_product_starts = (
    sales.groupby("product_id", as_index=False)
    .agg(observed_first_sale_date=("order_date", "min"))
)
product_start_dates = observed_product_starts.merge(
    products[["product_id", "launch_date"]],
    on="product_id",
    how="left",
    validate="1:1",
)
product_start_dates["effective_launch_date"] = product_start_dates[
    ["launch_date", "observed_first_sale_date"]
].min(axis=1)

observed_customer_starts = (
    sales.groupby("customer_id", as_index=False)
    .agg(observed_first_customer_sale=("order_date", "min"))
)
customer_start_dates = observed_customer_starts.merge(
    customers[["customer_id", "customer_since"]],
    on="customer_id",
    how="left",
    validate="1:1",
)
customer_start_dates["effective_customer_since"] = customer_start_dates[
    ["customer_since", "observed_first_customer_sale"]
].min(axis=1)

sales = sales.merge(
    product_start_dates[
        ["product_id", "observed_first_sale_date", "effective_launch_date"]
    ],
    on="product_id",
    how="left",
    validate="m:1",
).merge(
    customer_start_dates[
        [
            "customer_id", "observed_first_customer_sale",
            "effective_customer_since",
        ]
    ],
    on="customer_id",
    how="left",
    validate="m:1",
)

print(f"Valid canonical transactions: {len(sales):,}")
print(f"Rejected transactions:        {len(rejected_sales):,}")
display(sales.head(3))


Valid canonical transactions: 120,000
Rejected transactions:        0


,sales_id,order_id,date_id,order_date,year_month,customer_id,product_id,region_id,sales_rep_id,units,asp_eur,discount_pct,revenue_eur,revenue_local_currency,unit_cost_eur,gross_profit_eur,gross_margin_pct,is_outlier_order,cogs_eur,discount_value_eur,margin_category,order_size_category,missing_sales_rep_flag,missing_discount_flag,missing_margin_flag,units_sold,revenue,avg_discount_pct,transaction_id,region,country,currency,fx_to_eur,market_growth_factor,margin_factor,product_line,product_category,product_name,sku,lifecycle_stage,launch_date,is_declining_product,is_new_product,base_list_price_eur,base_unit_cost_eur,target_margin_pct,product_growth_factor,customer_segment,industry,customer_since,customer_size_score,base_churn_probability,churn_risk_band,seniority,annual_quota_eur,observed_first_sale_date,effective_launch_date,observed_first_customer_sale,effective_customer_since
0,1,ORD-00000001,20220304,2022-03-04,2022-03-01,282,1133,2,39,26,104.000,0.092,"2,704.130","2,704.130",78.470,663.790,0.245,False,"2,040.340",248.780,Low Margin,Medium Order,0,0,0,26,"2,704.130",0.092,1,Western Europe,France,EUR,1.000,1.020,0.970,Components,Connectivity Module,Connectivity Module Model 14,SKU-1133,Growth,2019-01-01,0,0,106.410,75.770,0.310,1.180,Mid-Market,Manufacturing,2022-11-10,3.527,0.067,Low Churn Risk,Professional,"1,229,114.000",2021-01-05,2019-01-01,2021-01-09,2021-01-09
1,2,ORD-00000002,20240503,2024-05-03,2024-05-01,562,1105,2,73,21,"1,698.070",0.089,"35,659.450","35,659.450",944.570,"15,823.420",0.444,False,"19,836.030","3,173.691",High Margin,Small Order,0,0,0,21,"35,659.450",0.089,2,Western Europe,France,EUR,1.000,1.020,0.970,Industrial Tech,Industrial Scanner,Industrial Scanner Model 06,SKU-1105,New,2023-01-01,0,1,"1,496.620",906.260,0.340,1.030,Mid-Market,Retail,2023-09-28,5.589,0.063,Low Churn Risk,Professional,"1,352,559.000",2021-01-09,2021-01-09,2021-01-13,2021-01-13
2,3,ORD-00000003,20241205,2024-12-05,2024-12-01,146,1007,7,41,117,983.150,0.123,"115,029.000","169,160.290",637.920,"40,392.640",0.351,False,"74,636.360","14,148.567",Medium Margin,Large Order,0,0,0,117,"115,029.000",0.123,3,North America,Canada,CAD,0.680,1.040,0.990,IT Devices,Laptop Pro,Laptop Pro Model 08,SKU-1007,Growth,2024-01-01,0,0,"1,017.790",613.650,0.290,1.080,Distributor,Technology,2021-06-22,10.633,0.060,Low Churn Risk,Professional,"1,276,718.000",2021-01-04,2021-01-04,2021-01-10,2021-01-10


## 6.1 Check transaction relationships and dates

Known product and customer dates are checked explicitly. Missing dimension matches are reported because the small sample files may not contain every referenced customer or product; full production extracts should normally have complete coverage.


In [11]:
relationship_checks = pd.DataFrame(
    {
        "relationship": [
            "sales → products",
            "sales → customers",
            "sales → regions",
            "sales → sales representatives (non-missing IDs)",
        ],
        "unmatched_rows": [
            int((~sales["product_id"].isin(products["product_id"])).sum()),
            int((~sales["customer_id"].isin(customers["customer_id"])).sum()),
            int((~sales["region_id"].isin(regions["region_id"])).sum()),
            int(
                (~sales.loc[sales["sales_rep_id"].notna(), "sales_rep_id"].isin(
                    sales_reps["sales_rep_id"]
                )).sum()
            ),
        ],
    }
)

known_launch = sales["launch_date"].notna()
sales_before_configured_launch = known_launch & (
    sales["order_date"] < sales["launch_date"]
)
known_customer_start = sales["customer_since"].notna()
sales_before_configured_customer_start = known_customer_start & (
    sales["order_date"] < sales["customer_since"]
)
sales_before_effective_launch = (
    sales["order_date"] < sales["effective_launch_date"]
)
sales_before_effective_customer_start = (
    sales["order_date"] < sales["effective_customer_since"]
)

time_checks = pd.DataFrame(
    {
        "rule": [
            "Sales before configured product launch",
            "Sales before configured customer start",
            "Sales before effective product launch",
            "Sales before effective customer start",
        ],
        "violating_rows": [
            int(sales_before_configured_launch.sum()),
            int(sales_before_configured_customer_start.sum()),
            int(sales_before_effective_launch.sum()),
            int(sales_before_effective_customer_start.sum()),
        ],
    }
)

display(relationship_checks)
display(time_checks)

if (
    sales_before_configured_launch.any()
    or sales_before_configured_customer_start.any()
):
    print(
        "Warning: configured start dates conflict with observed sales. "
        "Effective dates use the earlier value so valid history remains covered."
    )

assert not sales_before_effective_launch.any()
assert not sales_before_effective_customer_start.any()
print("Effective-date consistency checks passed.")


,relationship,unmatched_rows
0,sales → products,0
1,sales → customers,0
2,sales → regions,0
3,sales → sales representatives (non-missing IDs),0


,rule,violating_rows
0,Sales before configured product launch,19843
1,Sales before configured customer start,24318
2,Sales before effective product launch,0
3,Sales before effective customer start,0


Effective-date consistency checks passed.


In [12]:
sales_quality = pd.Series(
    {
        "rows": len(sales),
        "unique_transactions": sales["transaction_id"].nunique(),
        "start_month": sales["year_month"].min(),
        "end_month": sales["year_month"].max(),
        "negative_units": int((sales["units_sold"] < 0).sum()),
        "negative_revenue": int((sales["revenue"] < 0).sum()),
        "missing_sales_rep_ids": int(sales["sales_rep_id"].isna().sum()),
        "missing_discounts": int(sales["avg_discount_pct"].isna().sum()),
        "missing_margins": int(sales["gross_margin_pct"].isna().sum()),
    },
    name="value",
)
display(sales_quality.to_frame())

assert sales["transaction_id"].is_unique
assert not (sales["units_sold"] < 0).any()
assert not (sales["revenue"] < 0).any()


,value
rows,120000
unique_transactions,120000
start_month,2021-01-01 00:00:00
end_month,2025-12-01 00:00:00
negative_units,0
negative_revenue,0
missing_sales_rep_ids,730
missing_discounts,953
missing_margins,505


# 7. Aggregate sales to monthly analytical grains

The EWS model operates at **product × region × month**, not transaction level. We retain two outputs:

- `monthly_sales` for the product-risk panel;
- `customer_product_monthly` for customer drillthrough and recommendation use cases.

Revenue and units are reconciled back to the transaction table immediately after aggregation.


In [13]:
customer_product_monthly = (
    sales.groupby(
        ["year_month", "customer_id", "product_id", "region_id"],
        dropna=False,
        as_index=False,
    )
    .agg(
        units_sold=("units_sold", "sum"),
        revenue=("revenue", "sum"),
        order_count=("transaction_id", "nunique"),
        avg_discount_pct=("avg_discount_pct", "mean"),
        gross_profit_eur=("gross_profit_eur", "sum"),
        customer_segment=("customer_segment", most_common),
        churn_risk_band=("churn_risk_band", most_common),
    )
    .merge(
        regions[["region_id", "region"]],
        on="region_id",
        how="left",
        validate="m:1",
    )
)

customer_product_monthly = customer_product_monthly[
    [
        "year_month", "customer_id", "product_id", "units_sold",
        "revenue", "order_count", "avg_discount_pct", "region_id",
        "region", "gross_profit_eur", "customer_segment",
        "churn_risk_band",
    ]
]

sales_group_keys = ["product_id", "region_id", "year_month"]

monthly_sales = (
    sales.groupby(sales_group_keys, as_index=False)
    .agg(
        units_sold=("units_sold", "sum"),
        revenue=("revenue", "sum"),
        unique_customers=("customer_id", "nunique"),
        avg_discount_pct=("avg_discount_pct", "mean"),
        order_count=("transaction_id", "nunique"),
        active_sales_reps=("sales_rep_id", "nunique"),
        avg_asp_eur=("asp_eur", "mean"),
        gross_profit_eur=("gross_profit_eur", "sum"),
        cogs_eur=("cogs_eur", "sum"),
        discount_value_eur=("discount_value_eur", "sum"),
        outlier_order_count=("is_outlier_order", "sum"),
        missing_sales_rep_count=("missing_sales_rep_flag", "sum"),
        missing_discount_count=("missing_discount_flag", "sum"),
        missing_margin_count=("missing_margin_flag", "sum"),
        dominant_margin_category=("margin_category", most_common),
        dominant_order_size_category=("order_size_category", most_common),
        dominant_rep_seniority=("seniority", most_common),
    )
)
monthly_sales["gross_margin_pct"] = (
    monthly_sales["gross_profit_eur"]
    / monthly_sales["revenue"].replace(0, np.nan)
)


In [14]:
# Customer attributes are counted once per active customer per month.
active_customer_month = sales.drop_duplicates(
    sales_group_keys + ["customer_id"]
).copy()
active_customer_month["is_enterprise_customer"] = (
    active_customer_month["customer_segment"].eq("Enterprise").astype(int)
)
active_customer_month["is_high_churn_customer"] = (
    active_customer_month["churn_risk_band"]
    .astype("string")
    .str.contains("High", case=False, na=False)
    .astype(int)
)

customer_month_features = (
    active_customer_month.groupby(sales_group_keys, as_index=False)
    .agg(
        avg_customer_size_score=("customer_size_score", "mean"),
        avg_base_churn_probability=("base_churn_probability", "mean"),
        enterprise_customer_count=("is_enterprise_customer", "sum"),
        high_churn_customer_count=("is_high_churn_customer", "sum"),
    )
)

# Each active representative's annual quota is counted once per group-month.
representative_month = sales.dropna(subset=["sales_rep_id"]).drop_duplicates(
    sales_group_keys + ["sales_rep_id"]
)
quota_month_features = (
    representative_month.groupby(sales_group_keys, as_index=False)
    .agg(covered_annual_quota_eur=("annual_quota_eur", "sum"))
)

monthly_sales = monthly_sales.merge(
    customer_month_features,
    on=sales_group_keys,
    how="left",
    validate="1:1",
).merge(
    quota_month_features,
    on=sales_group_keys,
    how="left",
    validate="1:1",
)


In [15]:
reconciliation = pd.DataFrame(
    {
        "measure": ["units_sold", "revenue", "gross_profit_eur"],
        "transaction_total": [
            sales["units_sold"].sum(),
            sales["revenue"].sum(),
            sales["gross_profit_eur"].sum(),
        ],
        "monthly_total": [
            monthly_sales["units_sold"].sum(),
            monthly_sales["revenue"].sum(),
            monthly_sales["gross_profit_eur"].sum(),
        ],
    }
)
reconciliation["difference"] = (
    reconciliation["monthly_total"]
    - reconciliation["transaction_total"]
)

display(reconciliation)
display(monthly_sales.head(5))

assert np.allclose(reconciliation["difference"], 0, atol=0.01)
print("Transaction-to-month reconciliation passed.")


,measure,transaction_total,monthly_total,difference
0,units_sold,"4,759,689.000","4,759,689.000",0.000
1,revenue,"2,361,948,725.010","2,361,948,725.010",0.000
2,gross_profit_eur,"550,650,950.750","550,650,950.750",0.000


,product_id,region_id,year_month,units_sold,revenue,unique_customers,avg_discount_pct,order_count,active_sales_reps,avg_asp_eur,gross_profit_eur,cogs_eur,discount_value_eur,outlier_order_count,missing_sales_rep_count,missing_discount_count,missing_margin_count,dominant_margin_category,dominant_order_size_category,dominant_rep_seniority,gross_margin_pct,avg_customer_size_score,avg_base_churn_probability,enterprise_customer_count,high_churn_customer_count,covered_annual_quota_eur
0,1000,1,2021-01-01,81,"56,358.020",2,0.084,2,2,690.855,"12,852.590","43,505.430","4,824.214",0,0,0,0,Low Margin,Medium Order,Professional,0.228,6.149,0.045,1,0,"3,446,113.000"
1,1000,1,2021-03-01,16,"11,516.520",1,0.084,1,1,719.780,"2,968.630","8,547.890",967.388,0,0,0,0,Medium Margin,Small Order,Professional,0.258,7.370,0.083,0,0,"1,477,436.000"
2,1000,1,2021-04-01,35,"22,946.320",2,0.098,2,2,656.870,"4,313.720","18,632.600",581.655,0,0,1,0,Low Margin,Medium Order,Professional,0.188,7.302,0.079,0,0,"2,625,750.000"
3,1000,1,2021-06-01,160,"97,868.380",2,0.161,2,2,616.305,"12,906.330","84,962.050","15,919.751",0,0,0,0,Low Margin,Medium Order,Professional,0.132,8.924,0.030,2,0,"4,073,608.000"
4,1000,1,2021-07-01,9,"6,369.560",1,0.063,1,1,707.730,"1,397.200","4,972.360",401.282,0,0,0,0,Low Margin,Small Order,Professional,0.219,2.315,0.070,0,0,"1,160,163.000"


Transaction-to-month reconciliation passed.


# 8. Build the complete product–region–month panel

Transaction tables usually omit months with no sales. In an early-warning model, a zero-sales month is meaningful and must appear as an observation.

We therefore create all product–region–month combinations observed across the dimensions and fact tables. Months before a known product launch are excluded.


In [16]:
date_candidates = [sales["year_month"]]
date_source_columns = [
    (raw["inventory"], "year_month"),
    (raw["costs"], "year_month"),
    (raw["returns"], "return_date"),
    (raw["crm"], "activity_date"),
    (raw["pipeline"], "created_date"),
    (raw["market_activity"], "year_month"),
    (raw["market_signals"], "year_month"),
]
for dataframe, date_column in date_source_columns:
    date_candidates.append(to_month(dataframe[date_column]))

all_months_observed = pd.concat(date_candidates, ignore_index=True).dropna()
minimum_month = all_months_observed.min()
maximum_month = all_months_observed.max()
months = pd.date_range(minimum_month, maximum_month, freq="MS")

product_ids = pd.Index(products["product_id"].dropna().unique())
product_ids = product_ids.union(pd.Index(sales["product_id"].dropna().unique()))
product_ids = product_ids.union(
    pd.Index(normalize_id(raw["market_activity"]["product_id"]).dropna().unique())
)

region_ids = pd.Index(regions["region_id"].dropna().unique())
for table_name in ["sales", "inventory", "market_activity", "market_signals"]:
    region_ids = region_ids.union(
        pd.Index(normalize_id(raw[table_name]["region_id"]).dropna().unique())
    )

panel = pd.MultiIndex.from_product(
    [product_ids, region_ids, months],
    names=["product_id", "region_id", "year_month"],
).to_frame(index=False)

panel = panel.merge(
    products[product_dimension_columns],
    on="product_id",
    how="left",
    validate="m:1",
).merge(
    product_start_dates[
        ["product_id", "observed_first_sale_date", "effective_launch_date"]
    ],
    on="product_id",
    how="left",
    validate="m:1",
).merge(
    regions[region_dimension_columns],
    on="region_id",
    how="left",
    validate="m:1",
)

panel["product_line"] = panel["product_line"].fillna("Unknown")
panel["product_category"] = panel["product_category"].fillna("Unknown")
panel["region"] = panel["region"].fillna("Unknown")

# Products without observed sales fall back to the configured launch date.
panel["effective_launch_date"] = panel["effective_launch_date"].fillna(
    panel["launch_date"]
)
launch_month = (
    panel["effective_launch_date"].dt.to_period("M").dt.to_timestamp()
)
panel = panel.loc[
    panel["effective_launch_date"].isna()
    | (panel["year_month"] >= launch_month)
].copy()

modeling = panel.merge(
    monthly_sales,
    on=["product_id", "region_id", "year_month"],
    how="left",
    validate="1:1",
)


In [17]:
zero_when_no_sales = [
    "units_sold", "revenue", "unique_customers", "order_count",
    "active_sales_reps", "gross_profit_eur", "cogs_eur",
    "discount_value_eur", "outlier_order_count",
    "missing_sales_rep_count", "missing_discount_count",
    "missing_margin_count", "enterprise_customer_count",
    "high_churn_customer_count", "covered_annual_quota_eur",
]
modeling[zero_when_no_sales] = modeling[zero_when_no_sales].fillna(0)
modeling["no_sales_flag"] = modeling["units_sold"].eq(0).astype(int)

# Ratios and averages remain missing when there was no transaction.
panel_check = pd.Series(
    {
        "panel_rows": len(modeling),
        "duplicate_panel_keys": int(
            modeling.duplicated(
                ["product_id", "region_id", "year_month"]
            ).sum()
        ),
        "zero_sales_rows": int(modeling["units_sold"].eq(0).sum()),
        "zero_sales_share_pct": round(
            modeling["units_sold"].eq(0).mean() * 100, 2
        ),
        "start_month": modeling["year_month"].min(),
        "end_month": modeling["year_month"].max(),
    },
    name="value",
)
display(panel_check.to_frame())

example_key = modeling[["product_id", "region_id"]].iloc[0]
example_panel = modeling.loc[
    modeling["product_id"].eq(example_key["product_id"])
    & modeling["region_id"].eq(example_key["region_id"]),
    ["product_id", "region", "year_month", "units_sold", "revenue"],
].head(12)
display(example_panel)

assert panel_check["duplicate_panel_keys"] == 0


,value
panel_rows,120000
duplicate_panel_keys,0
zero_sales_rows,50148
zero_sales_share_pct,41.790
start_month,2021-01-01 00:00:00
end_month,2025-12-01 00:00:00


,product_id,region,year_month,units_sold,revenue
0,1000,DACH,2021-01-01,81.000,"56,358.020"
1,1000,DACH,2021-02-01,0.000,0.000
2,1000,DACH,2021-03-01,16.000,"11,516.520"
3,1000,DACH,2021-04-01,35.000,"22,946.320"
4,1000,DACH,2021-05-01,0.000,0.000
5,1000,DACH,2021-06-01,160.000,"97,868.380"
6,1000,DACH,2021-07-01,9.000,"6,369.560"
7,1000,DACH,2021-08-01,0.000,0.000
8,1000,DACH,2021-09-01,0.000,0.000
9,1000,DACH,2021-10-01,125.000,"84,072.480"


# 9. Add operational and commercial context

Each source is standardized and aggregated to a grain compatible with the panel before joining. `validate=` prevents accidental row multiplication.


## 9.1 Inventory and supply features

Inventory is already at month × product × region. We keep stock balances, stockout indicators, inventory value, and derive supply coverage and a transparent backorder proxy.


In [18]:
inventory = raw["inventory"].copy()
inventory["product_id"] = normalize_id(inventory["product_id"])
inventory["region_id"] = normalize_id(inventory["region_id"])
inventory["year_month"] = to_month(inventory["year_month"])
inventory["stockout_flag"] = numeric(
    inventory, "stockout_flag", 0
).fillna(0).astype(int)
inventory["zero_stock_flag"] = numeric(
    inventory, "zero_stock_flag", 0
).fillna(0).astype(int)

inventory_monthly = (
    inventory.groupby(
        ["product_id", "region_id", "year_month"], as_index=False
    )
    .agg(
        opening_stock_units=("opening_stock_units", "sum"),
        production_units=("production_units", "sum"),
        ending_stock_units=("ending_stock_units", "sum"),
        stockout_flag=("stockout_flag", "max"),
        zero_stock_flag=("zero_stock_flag", "max"),
        inventory_value_eur=("inventory_value_eur", "sum"),
    )
)

rows_before_merge = len(modeling)
modeling = modeling.merge(
    inventory_monthly,
    on=["product_id", "region_id", "year_month"],
    how="left",
    validate="1:1",
)
assert len(modeling) == rows_before_merge

modeling["inventory_available_flag"] = modeling[
    "opening_stock_units"
].notna().astype(int)
available_supply = (
    modeling["opening_stock_units"].fillna(0)
    + modeling["production_units"].fillna(0)
)
modeling["backorder_units"] = np.where(
    modeling["inventory_available_flag"].eq(1),
    (modeling["units_sold"] - available_supply).clip(lower=0),
    0,
)
modeling["supply_coverage_ratio"] = (
    available_supply / modeling["units_sold"].replace(0, np.nan)
)
modeling["stockout_flag"] = modeling["stockout_flag"].fillna(0).astype(int)
modeling["zero_stock_flag"] = modeling["zero_stock_flag"].fillna(0).astype(int)

display(modeling[
    [
        "product_id", "region", "year_month", "units_sold",
        "opening_stock_units", "production_units",
        "ending_stock_units", "stockout_flag", "backorder_units",
    ]
].head())


,product_id,region,year_month,units_sold,opening_stock_units,production_units,ending_stock_units,stockout_flag,backorder_units
0,1000,DACH,2021-01-01,81.000,53.000,79.000,51.000,0,0.000
1,1000,DACH,2021-02-01,0.000,NaN,NaN,NaN,0,0.000
2,1000,DACH,2021-03-01,16.000,23.000,18.000,25.000,0,0.000
3,1000,DACH,2021-04-01,35.000,47.000,28.000,40.000,0,0.000
4,1000,DACH,2021-05-01,0.000,NaN,NaN,NaN,0,0.000


## 9.2 Costs, returns and calendar features

Costs join by product and month because the cost source has no region key. Returns join by product, region, and return month. Calendar attributes are taken from the date dimension, with year/month fallbacks derived from `year_month` when a sample extract contains only part of the date table.


In [19]:
costs = raw["costs"].copy()
costs["product_id"] = normalize_id(costs["product_id"])
costs["year_month"] = to_month(costs["year_month"])
cost_monthly = (
    costs.groupby(["product_id", "year_month"], as_index=False)
    .agg(
        standard_unit_cost_eur=("standard_unit_cost_eur", "mean"),
        actual_unit_cost_eur=("actual_unit_cost_eur", "mean"),
        cost_variance_eur=("cost_variance_eur", "mean"),
        cost_variance_pct=("cost_variance_pct", "mean"),
    )
)

modeling = modeling.merge(
    cost_monthly,
    on=["product_id", "year_month"],
    how="left",
    validate="m:1",
)

returns = raw["returns"].copy()
returns["product_id"] = normalize_id(returns["product_id"])
returns["region_id"] = normalize_id(returns["region_id"])
returns["year_month"] = to_month(returns["return_date"])
returns_monthly = (
    returns.groupby(
        ["product_id", "region_id", "year_month"], as_index=False
    )
    .agg(
        return_units=("return_units", "sum"),
        return_value_eur=("return_value_eur", "sum"),
        return_count=("return_id", "nunique"),
    )
)
modeling = modeling.merge(
    returns_monthly,
    on=["product_id", "region_id", "year_month"],
    how="left",
    validate="1:1",
)
return_columns = ["return_units", "return_value_eur", "return_count"]
modeling[return_columns] = modeling[return_columns].fillna(0)
modeling["return_rate_units"] = (
    modeling["return_units"]
    / modeling["units_sold"].replace(0, np.nan)
)


In [20]:
dates = raw["date"].copy()
date_source_column = (
    "MonthStartDate" if "MonthStartDate" in dates else "Date"
)
dates["year_month"] = to_month(dates[date_source_column])
date_monthly = dates.sort_values("year_month").drop_duplicates("year_month")
calendar_columns = existing(
    date_monthly,
    [
        "year_month", "Year", "Quarter", "QuarterName", "Month",
        "MonthName", "MonthShortName", "YearMonthKey",
        "MonthStartDate", "MonthEndDate",
    ],
)
modeling = modeling.merge(
    date_monthly[calendar_columns],
    on="year_month",
    how="left",
    validate="m:1",
)

# Fallbacks keep the calendar usable with partial sample extracts.
modeling["Year"] = modeling["Year"].fillna(modeling["year_month"].dt.year)
modeling["Quarter"] = modeling["Quarter"].fillna(
    modeling["year_month"].dt.quarter
)
modeling["Month"] = modeling["Month"].fillna(modeling["year_month"].dt.month)
modeling["MonthName"] = modeling["MonthName"].fillna(
    modeling["year_month"].dt.month_name()
)
modeling["YearMonthKey"] = modeling["YearMonthKey"].fillna(
    modeling["year_month"].dt.strftime("%Y%m").astype(int)
)

display(modeling[
    [
        "year_month", "Year", "Quarter", "Month", "MonthName",
        "actual_unit_cost_eur", "return_units", "return_rate_units",
    ]
].head())


,year_month,Year,Quarter,Month,MonthName,actual_unit_cost_eur,return_units,return_rate_units
0,2021-01-01,2021,1,1,January,573.850,0.000,0.000
1,2021-02-01,2021,1,2,February,562.050,0.000,NaN
2,2021-03-01,2021,1,3,March,570.440,0.000,0.000
3,2021-04-01,2021,2,4,April,575.790,0.000,0.000
4,2021-05-01,2021,2,5,May,558.630,0.000,NaN


## 9.3 CRM and pipeline features

CRM activities do not contain a product key, so they are aggregated at region-month level. Pipeline opportunities contain a product group and therefore join at month × region × product category.

Customer region is preferred. Sales-representative region is used as a fallback when a partial customer extract does not contain the referenced customer.


In [21]:
customer_region_lookup = customers[["customer_id", "region_id"]].rename(
    columns={"region_id": "customer_region_id"}
)
representative_region_lookup = sales_reps[
    ["sales_rep_id", "region_id"]
].rename(columns={"region_id": "representative_region_id"})

crm = raw["crm"].copy()
crm["customer_id"] = normalize_id(crm["customer_id"])
crm["sales_rep_id"] = normalize_id(crm["sales_rep_id"])
crm["year_month"] = to_month(crm["activity_date"])
crm = crm.merge(
    customer_region_lookup,
    on="customer_id",
    how="left",
    validate="m:1",
).merge(
    representative_region_lookup,
    on="sales_rep_id",
    how="left",
    validate="m:1",
)
crm["region_id"] = crm["customer_region_id"].combine_first(
    crm["representative_region_id"]
)
crm["is_demo_like"] = (
    crm["activity_type"]
    .astype("string")
    .str.contains(
        "demo|visit|meeting|presentation", case=False, na=False
    )
    .astype(int)
)

crm_region_month = (
    crm.dropna(subset=["year_month", "region_id"])
    .groupby(["year_month", "region_id"], as_index=False)
    .agg(
        crm_activity_count=("activity_id", "nunique"),
        crm_activity_minutes=("activity_minutes", "sum"),
        crm_demo_like_count=("is_demo_like", "sum"),
        avg_sentiment_score=("sentiment_score", "mean"),
        avg_customer_health_score=("customer_health_score", "mean"),
    )
)
modeling = modeling.merge(
    crm_region_month,
    on=["year_month", "region_id"],
    how="left",
    validate="m:1",
)


In [22]:
pipeline = raw["pipeline"].copy()
pipeline["customer_id"] = normalize_id(pipeline["customer_id"])
pipeline["sales_rep_id"] = normalize_id(pipeline["sales_rep_id"])
pipeline["year_month"] = to_month(pipeline["created_date"])
pipeline["product_category"] = pipeline["product_group"].fillna("Unknown")
pipeline = pipeline.merge(
    customer_region_lookup,
    on="customer_id",
    how="left",
    validate="m:1",
).merge(
    representative_region_lookup,
    on="sales_rep_id",
    how="left",
    validate="m:1",
)
pipeline["region_id"] = pipeline["customer_region_id"].combine_first(
    pipeline["representative_region_id"]
)

pipeline_monthly = (
    pipeline.dropna(subset=["year_month", "region_id"])
    .groupby(
        ["year_month", "region_id", "product_category"],
        as_index=False,
    )
    .agg(
        pipeline_opportunities=("opportunity_id", "nunique"),
        weighted_pipeline_eur=("weighted_pipeline_eur", "sum"),
        expected_pipeline_eur=("expected_value_eur", "sum"),
        avg_win_probability=("win_probability", "mean"),
        avg_days_to_close=("days_to_close", "mean"),
        closed_won_count=("is_closed_won", "sum"),
        closed_lost_count=("is_closed_lost", "sum"),
    )
)
modeling = modeling.merge(
    pipeline_monthly,
    on=["year_month", "region_id", "product_category"],
    how="left",
    validate="m:1",
)

activity_zero_columns = [
    "crm_activity_count", "crm_activity_minutes", "crm_demo_like_count",
    "pipeline_opportunities", "weighted_pipeline_eur",
    "expected_pipeline_eur", "closed_won_count", "closed_lost_count",
]
modeling[activity_zero_columns] = modeling[activity_zero_columns].fillna(0)
modeling["pipeline_conversion_rate"] = (
    modeling["closed_won_count"]
    / (
        modeling["closed_won_count"] + modeling["closed_lost_count"]
    ).replace(0, np.nan)
)

display(modeling[
    [
        "year_month", "region", "product_category",
        "crm_activity_count", "avg_sentiment_score",
        "pipeline_opportunities", "weighted_pipeline_eur",
    ]
].head())


,year_month,region,product_category,crm_activity_count,avg_sentiment_score,pipeline_opportunities,weighted_pipeline_eur
0,2021-01-01,DACH,Laptop Pro,122,0.044,3.000,"80,568.080"
1,2021-02-01,DACH,Laptop Pro,109,0.074,4.000,"127,680.470"
2,2021-03-01,DACH,Laptop Pro,124,0.041,3.000,"62,071.030"
3,2021-04-01,DACH,Laptop Pro,116,0.144,2.000,"24,017.380"
4,2021-05-01,DACH,Laptop Pro,106,0.101,5.000,"112,426.580"


# 10. Add the new market datasets

- `market_activity` supplies campaign and website metrics at product-region-month grain.
- `market_signals` supplies demand, competition, macro, supply and opportunity metrics at product-family/product-group-region-month grain.

Availability flags distinguish a genuine zero from a missing enrichment row.

## 10.1 Market activity

The source is aggregated defensively even though its expected grain is already unique. Click-through rate and cost per lead are recalculated from the aggregated totals so they remain mathematically consistent.


In [23]:
market_activity = raw["market_activity"].copy()
market_activity["product_id"] = normalize_id(
    market_activity["product_id"]
)
market_activity["region_id"] = normalize_id(
    market_activity["region_id"]
)
market_activity["year_month"] = to_month(
    market_activity["year_month"]
)

market_activity_monthly = (
    market_activity.groupby(
        ["product_id", "region_id", "year_month"], as_index=False
    )
    .agg(
        campaign_flag=("campaign_flag", "max"),
        campaign_channel=("campaign_channel", most_common),
        campaign_spend_eur=("campaign_spend_eur", "sum"),
        campaign_impressions=("campaign_impressions", "sum"),
        campaign_clicks=("campaign_clicks", "sum"),
        website_visits=("website_visits", "sum"),
        product_page_views=("product_page_views", "sum"),
        demo_requests=("demo_requests", "sum"),
        marketing_qualified_leads=(
            "marketing_qualified_leads", "sum"
        ),
        regional_crm_activity_index=(
            "regional_crm_activity_index", "mean"
        ),
        activity_pipeline_interest_index=(
            "pipeline_interest_index", "mean"
        ),
    )
)
market_activity_monthly["campaign_ctr_pct"] = (
    market_activity_monthly["campaign_clicks"]
    / market_activity_monthly["campaign_impressions"].replace(0, np.nan)
    * 100
).fillna(0)
market_activity_monthly["cost_per_lead_eur"] = (
    market_activity_monthly["campaign_spend_eur"]
    / market_activity_monthly["marketing_qualified_leads"].replace(0, np.nan)
)

market_activity_monthly["market_activity_available_flag"] = 1
rows_before_merge = len(modeling)
modeling = modeling.merge(
    market_activity_monthly,
    on=["product_id", "region_id", "year_month"],
    how="left",
    validate="1:1",
)
assert len(modeling) == rows_before_merge

modeling["market_activity_available_flag"] = modeling[
    "market_activity_available_flag"
].fillna(0).astype(int)


## 10.2 Market signals

Product family and product group are mapped to the canonical EWS names `product_line` and `product_category`. The pipeline-interest column is renamed so it cannot be confused with the similarly named market-activity metric.


In [24]:
market_signals = raw["market_signals"].copy()
market_signals["region_id"] = normalize_id(market_signals["region_id"])
market_signals["year_month"] = to_month(market_signals["year_month"])
market_signals["product_line"] = market_signals["product_family"].fillna(
    "Unknown"
)
market_signals["product_category"] = market_signals[
    "product_group"
].fillna("Unknown")

market_signals_monthly = (
    market_signals.groupby(
        [
            "year_month", "product_line", "product_category",
            "region_id",
        ],
        as_index=False,
    )
    .agg(
        market_demand_index=("market_demand_index", "mean"),
        market_growth_pct=("market_growth_pct", "mean"),
        competitor_pressure_index=(
            "competitor_pressure_index", "mean"
        ),
        seasonality_index=("seasonality_index", "mean"),
        macro_business_index=("macro_business_index", "mean"),
        supply_pressure_index=("supply_pressure_index", "mean"),
        market_pipeline_interest_index=(
            "pipeline_interest_index", "mean"
        ),
        demand_shock_flag=("demand_shock_flag", "max"),
        market_opportunity_score=("market_opportunity_score", "mean"),
        regional_market_growth_factor=(
            "regional_market_growth_factor", "mean"
        ),
    )
)
market_signals_monthly["market_signal_available_flag"] = 1

rows_before_merge = len(modeling)
modeling = modeling.merge(
    market_signals_monthly,
    on=[
        "year_month", "product_line", "product_category", "region_id"
    ],
    how="left",
    validate="m:1",
)
assert len(modeling) == rows_before_merge

modeling["market_signal_available_flag"] = modeling[
    "market_signal_available_flag"
].fillna(0).astype(int)


In [25]:
enrichment_coverage_columns = [
    "opening_stock_units",
    "actual_unit_cost_eur",
    "return_units",
    "crm_activity_count",
    "pipeline_opportunities",
    "campaign_flag",
    "market_demand_index",
]
enrichment_coverage = pd.DataFrame(
    {
        "column": enrichment_coverage_columns,
        "available_rows_pct": [
            round(modeling[column].notna().mean() * 100, 2)
            for column in enrichment_coverage_columns
        ],
        "non_zero_rows_pct": [
            round(modeling[column].fillna(0).ne(0).mean() * 100, 2)
            for column in enrichment_coverage_columns
        ],
    }
)

market_join_summary = pd.DataFrame(
    {
        "enrichment": ["market_activity", "market_signals"],
        "available_rows": [
            int(modeling["market_activity_available_flag"].sum()),
            int(modeling["market_signal_available_flag"].sum()),
        ],
        "coverage_pct": [
            round(modeling["market_activity_available_flag"].mean() * 100, 2),
            round(modeling["market_signal_available_flag"].mean() * 100, 2),
        ],
    }
)

display(enrichment_coverage)
display(market_join_summary)
display(modeling[
    [
        "product_id", "region", "year_month", "campaign_flag",
        "website_visits", "market_demand_index",
        "competitor_pressure_index", "market_opportunity_score",
    ]
].head())


,column,available_rows_pct,non_zero_rows_pct
0,opening_stock_units,58.210,58.140
1,actual_unit_cost_eur,100.000,100.000
2,return_units,100.000,3.420
3,crm_activity_count,100.000,100.000
4,pipeline_opportunities,100.000,87.650
5,campaign_flag,100.000,18.280
6,market_demand_index,100.000,100.000


,enrichment,available_rows,coverage_pct
0,market_activity,120000,100.000
1,market_signals,120000,100.000


,product_id,region,year_month,campaign_flag,website_visits,market_demand_index,competitor_pressure_index,market_opportunity_score
0,1000,DACH,2021-01-01,0,270,97.400,61.320,79.240
1,1000,DACH,2021-02-01,0,264,102.270,62.950,81.020
2,1000,DACH,2021-03-01,1,3137,100.950,59.960,80.390
3,1000,DACH,2021-04-01,0,242,109.600,53.590,81.590
4,1000,DACH,2021-05-01,1,1634,118.380,54.440,89.200


# 11. Create time-based features

Rows are sorted within each product-region history. Lags and rolling statistics use `.shift(1)` so the current value is excluded from its own historical baseline.

The slope helper fits a simple line to a short rolling window. Its output is positive for an upward trend and negative for a downward trend.


In [26]:
def slope(values: np.ndarray) -> float:
    values = np.asarray(values, dtype=float)
    if len(values) < 2 or np.all(np.isnan(values)):
        return np.nan
    return float(np.polyfit(np.arange(len(values)), values, 1)[0])

#consecutive decline flag over 3 months
def consecutive_decline(series):
    return int((series.diff().dropna() < 0).all())

In [27]:
modeling = modeling.sort_values(
    ["product_id", "region_id", "year_month"]
).reset_index(drop=True)
product_region_history = modeling.groupby(
    ["product_id", "region_id"], group_keys=False
)

modeling["product_age_months"] = (
    (
        modeling["year_month"].dt.year
        - modeling["effective_launch_date"].dt.year
    )
    * 12
    + modeling["year_month"].dt.month
    - modeling["effective_launch_date"].dt.month
).clip(lower=0)

modeling["units_lag_1m"] = product_region_history["units_sold"].shift(1)
modeling["units_lag_3m"] = product_region_history["units_sold"].shift(3)
modeling["revenue_lag_1m"] = product_region_history["revenue"].shift(1)
modeling["customers_lag_1m"] = product_region_history[
    "unique_customers"
].shift(1)

modeling["rolling_units_mean_3m"] = product_region_history[
    "units_sold"
].transform(lambda series: series.shift(1).rolling(3, min_periods=2).mean())
modeling["rolling_units_mean_6m"] = product_region_history[
    "units_sold"
].transform(lambda series: series.shift(1).rolling(6, min_periods=3).mean())
modeling["rolling_units_std_3m"] = product_region_history[
    "units_sold"
].transform(lambda series: series.shift(1).rolling(3, min_periods=2).std())
modeling["volatility_3m"] = (
    modeling["rolling_units_std_3m"]
    / modeling["rolling_units_mean_3m"].replace(0, np.nan)
)

modeling["mom_units_growth_pct"] = (
    (modeling["units_sold"] - modeling["units_lag_1m"])
    / modeling["units_lag_1m"].replace(0, np.nan)
)
modeling["customer_count_growth_pct"] = (
    (modeling["unique_customers"] - modeling["customers_lag_1m"])
    / modeling["customers_lag_1m"].replace(0, np.nan)
)
modeling["trend_slope_3m"] = product_region_history[
    "units_sold"
].transform(
    lambda series: series.shift(1).rolling(3, min_periods=3).apply(
        slope, raw=True
    )
)
modeling["trend_slope_6m"] = product_region_history[
    "units_sold"
].transform(
    lambda series: series.shift(1).rolling(6, min_periods=6).apply(
        slope, raw=True
    )
)

modeling["consec_drop_3m"] = product_region_history[
    "units_sold"
].transform(
    lambda series: series.shift(1).rolling(4, min_periods=4).apply(
        consecutive_decline, raw=False
    )
)


In [28]:
expected_units = (
    modeling["rolling_units_mean_3m"]
    + modeling["trend_slope_3m"].fillna(0)
)
modeling["risk_factor_sales_drop"] = (
    modeling["consec_drop_3m"] == 1).astype(int)
modeling["risk_factor_customer_drop"] = (
    modeling["customer_count_growth_pct"] < -0.25
).astype(int)

volatility_75th = modeling["volatility_3m"].quantile(0.75)

modeling["risk_factor_volatility"] = (
    modeling["volatility_3m"] > volatility_75th
).astype(int)
modeling["risk_factor_under_trend"] = (
    (modeling["units_sold"] < expected_units * 0.75)
    & (expected_units > 0)
).astype(int)

feature_example = modeling[
    [
        "product_id", "region", "year_month", "units_sold",
        "units_lag_1m", "rolling_units_mean_3m",
        "mom_units_growth_pct", "trend_slope_3m",
        "risk_factor_sales_drop", "risk_factor_under_trend",
    ]
].head(12)
display(feature_example)


,product_id,region,year_month,units_sold,units_lag_1m,rolling_units_mean_3m,mom_units_growth_pct,trend_slope_3m,risk_factor_sales_drop,risk_factor_under_trend
0,1000,DACH,2021-01-01,81.000,NaN,NaN,NaN,NaN,0,0
1,1000,DACH,2021-02-01,0.000,81.000,NaN,-1.000,NaN,0,0
2,1000,DACH,2021-03-01,16.000,0.000,40.500,NaN,NaN,0,1
3,1000,DACH,2021-04-01,35.000,16.000,32.333,1.188,-32.500,0,0
4,1000,DACH,2021-05-01,0.000,35.000,17.000,-1.000,17.500,0,1
5,1000,DACH,2021-06-01,160.000,0.000,17.000,NaN,-8.000,0,0
6,1000,DACH,2021-07-01,9.000,160.000,65.000,-0.944,62.500,0,1
7,1000,DACH,2021-08-01,0.000,9.000,56.333,-1.000,4.500,0,1
8,1000,DACH,2021-09-01,0.000,0.000,56.333,NaN,-80.000,0,0
9,1000,DACH,2021-10-01,125.000,0.000,3.000,NaN,-4.500,0,0


# 12. Create the one-month-ahead target

The target is based on next-month outcomes. A row is labeled:

- **High risk (2):** at least three future triggers;
- **Medium risk (1):** one or two future triggers;
- **Low risk (0):** no future trigger.

The five triggers are next-month sales decline, customer decline, high volatility, under-trend performance, and stockout. The final month of every product-region history has no observable next month and remains unlabeled.


In [29]:
modeling["future_units"] = product_region_history["units_sold"].shift(-1)
modeling["future_customers"] = product_region_history[
    "unique_customers"
].shift(-1)
modeling["future_volatility"] = product_region_history[
    "volatility_3m"
].shift(-1)
modeling["future_expected_units"] = (
    product_region_history["rolling_units_mean_3m"].shift(-1)
    + product_region_history["trend_slope_3m"].shift(-1).fillna(0)
)
modeling["future_stockout"] = product_region_history[
    "stockout_flag"
].shift(-1)

modeling["future_sales_drop"] = product_region_history[
    "consec_drop_3m"
].shift(-1)

modeling["future_sales_drop_trigger"] = modeling["future_sales_drop"].fillna(0).astype(int)
modeling["future_customer_drop_trigger"] = (
    modeling["future_customers"]
    < modeling["unique_customers"] * 0.75
).astype(int)
modeling["future_volatility_trigger"] = (
    modeling["future_volatility"] >= volatility_75th
).astype(int)
modeling["future_under_trend_trigger"] = (
    (modeling["future_units"] < modeling["future_expected_units"] * 0.72)
    & (modeling["future_expected_units"] > 0)
).astype(int)
modeling["future_stockout_trigger"] = modeling[
    "future_stockout"
].fillna(0).astype(int)

modeling["target_risk_score"] = modeling[
    [
        "future_sales_drop_trigger",
        "future_customer_drop_trigger",
        "future_volatility_trigger",
        "future_under_trend_trigger",
        "future_stockout_trigger",
    ]
].sum(axis=1)
modeling[TARGET_COL] = np.select(
    [modeling["target_risk_score"] >= 3, modeling["target_risk_score"] >= 2],
    [2, 1],
    default=0,
).astype(float)

final_group_row = product_region_history.cumcount(ascending=False).eq(0)
modeling.loc[final_group_row, TARGET_COL] = np.nan
modeling["target_month"] = modeling["year_month"] + pd.offsets.MonthBegin(1)

target_summary = (
    modeling[TARGET_COL]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("risk_label")
    .reset_index(name="rows")
)
target_summary["share_pct"] = (
    target_summary["rows"] / len(modeling) * 100
).round(2)
display(target_summary)


,risk_label,rows,share_pct
0,0.000,79313,66.090
1,1.000,33349,27.790
2,2.000,5338,4.450
3,NaN,2000,1.670


# 13. Create the compatible and enriched output datasets

`ews_modeling_dataset.csv` preserves the original 32-column EWS contract exactly. The existing fields `campaign_flag`, `website_visits`, `demo_requests`, `market_demand_index`, and `competitor_pressure_index` now come from the dedicated market tables rather than internally derived proxies.

`ews_modeling_dataset_enriched.csv` starts with the same 32 columns and then adds the new operational, financial, customer, marketing, and market features.


In [30]:
EWS_COLUMNS = [
    "product_id", "year_month", "product_line", "product_category",
    "region", "units_sold", "revenue", "unique_customers",
    "avg_discount_pct", "units_lag_1m", "units_lag_3m",
    "revenue_lag_1m", "rolling_units_mean_3m",
    "rolling_units_mean_6m", "rolling_units_std_3m",
    "volatility_3m", "mom_units_growth_pct",
    "customer_count_growth_pct", "trend_slope_3m",
    "trend_slope_6m", "stockout_flag", "backorder_units",
    "market_demand_index", "competitor_pressure_index",
    "campaign_flag", "website_visits", "demo_requests",
    "risk_factor_sales_drop", "risk_factor_customer_drop",
    "risk_factor_volatility", "risk_factor_under_trend",
    "next_month_risk_label",
]

missing_compatible_columns = [
    column for column in EWS_COLUMNS if column not in modeling.columns
]
if missing_compatible_columns:
    raise AssertionError(
        f"Missing original EWS columns: {missing_compatible_columns}"
    )

future_helper_columns = [
    column
    for column in modeling.columns
    if column.startswith("future_") or column == "target_risk_score"
]

final_modeling_dataset = modeling[EWS_COLUMNS].copy()
enriched_extra_columns = [
    column
    for column in modeling.columns
    if column not in EWS_COLUMNS and column not in future_helper_columns
]
enriched_modeling_dataset = modeling[
    EWS_COLUMNS + enriched_extra_columns
].copy()

for dataframe in [final_modeling_dataset, enriched_modeling_dataset]:
    dataframe["year_month"] = dataframe["year_month"].dt.strftime("%Y-%m")
    numeric_columns = dataframe.select_dtypes(include=[np.number]).columns
    dataframe[numeric_columns] = dataframe[numeric_columns].replace(
        [np.inf, -np.inf], np.nan
    )

enriched_modeling_dataset["target_month"] = pd.to_datetime(
    enriched_modeling_dataset["target_month"]
).dt.strftime("%Y-%m")

customer_product_monthly["year_month"] = customer_product_monthly[
    "year_month"
].dt.strftime("%Y-%m")

print(
    f"Compatible EWS dataset: {final_modeling_dataset.shape[0]:,} rows "
    f"x {final_modeling_dataset.shape[1]} columns"
)
print(
    f"Enriched EWS dataset:   {enriched_modeling_dataset.shape[0]:,} rows "
    f"x {enriched_modeling_dataset.shape[1]} columns"
)
display(final_modeling_dataset.head())


Compatible EWS dataset: 120,000 rows x 32 columns
Enriched EWS dataset:   120,000 rows x 131 columns


,product_id,year_month,product_line,product_category,region,units_sold,revenue,unique_customers,avg_discount_pct,units_lag_1m,units_lag_3m,revenue_lag_1m,rolling_units_mean_3m,rolling_units_mean_6m,rolling_units_std_3m,volatility_3m,mom_units_growth_pct,customer_count_growth_pct,trend_slope_3m,trend_slope_6m,stockout_flag,backorder_units,market_demand_index,competitor_pressure_index,campaign_flag,website_visits,demo_requests,risk_factor_sales_drop,risk_factor_customer_drop,risk_factor_volatility,risk_factor_under_trend,next_month_risk_label
0,1000,2021-01,IT Devices,Laptop Pro,DACH,81.000,"56,358.020",2.000,0.084,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.000,97.400,61.320,0,270,1,0,0,0,0,0.000
1,1000,2021-02,IT Devices,Laptop Pro,DACH,0.000,0.000,0.000,NaN,81.000,NaN,"56,358.020",NaN,NaN,NaN,NaN,-1.000,-1.000,NaN,NaN,0,0.000,102.270,62.950,0,264,2,0,1,0,0,0.000
2,1000,2021-03,IT Devices,Laptop Pro,DACH,16.000,"11,516.520",1.000,0.084,0.000,NaN,0.000,40.500,NaN,57.276,1.414,NaN,NaN,NaN,NaN,0,0.000,100.950,59.960,1,3137,36,0,0,0,1,0.000
3,1000,2021-04,IT Devices,Laptop Pro,DACH,35.000,"22,946.320",2.000,0.098,16.000,81.000,"11,516.520",32.333,32.333,42.899,1.327,1.188,1.000,-32.500,NaN,0,0.000,109.600,53.590,0,242,4,0,0,0,0,1.000
4,1000,2021-05,IT Devices,Laptop Pro,DACH,0.000,0.000,0.000,NaN,35.000,0.000,"22,946.320",17.000,33.000,17.521,1.031,-1.000,-1.000,17.500,NaN,0,0.000,118.380,54.440,1,1634,20,0,1,0,1,0.000


## 13.1 Confirm that every source contributes
Each source must contribute at least one field to the enriched modeling dataset.

In [31]:
SOURCE_FEATURE_MAP = {
    "sales": ["units_sold", "revenue", "gross_profit_eur"],
    "products": ["product_line", "lifecycle_stage", "target_margin_pct"],
    "customers": ["avg_base_churn_probability", "high_churn_customer_count"],
    "regions": ["region", "market_growth_factor", "margin_factor"],
    "sales_reps": ["active_sales_reps", "covered_annual_quota_eur"],
    "inventory": ["stockout_flag", "ending_stock_units", "inventory_value_eur"],
    "costs": ["actual_unit_cost_eur", "cost_variance_pct"],
    "returns": ["return_units", "return_rate_units"],
    "crm": ["crm_activity_count", "avg_sentiment_score"],
    "pipeline": ["pipeline_opportunities", "weighted_pipeline_eur"],
    "date": ["Year", "Quarter", "Month"],
    "market_activity": ["campaign_spend_eur", "website_visits"],
    "market_signals": ["market_demand_index", "market_opportunity_score"],
}

source_usage_rows = []
for source_table, contributed_columns in SOURCE_FEATURE_MAP.items():
    missing_columns = [
        column
        for column in contributed_columns
        if column not in enriched_modeling_dataset.columns
    ]
    source_usage_rows.append(
        {
            "source_table": source_table,
            "contributed_columns": ", ".join(contributed_columns),
            "missing_columns": ", ".join(missing_columns) or "None",
            "used": not missing_columns,
        }
    )

source_usage = pd.DataFrame(source_usage_rows)
display(source_usage)
assert source_usage["used"].all()


,source_table,contributed_columns,missing_columns,used
0,sales,"units_sold, revenue, gross_profit_eur",None,True
1,products,"product_line, lifecycle_stage, target_margin_pct",None,True
2,customers,"avg_base_churn_probability, high_churn_custome...",None,True
3,regions,"region, market_growth_factor, margin_factor",None,True
4,sales_reps,"active_sales_reps, covered_annual_quota_eur",None,True
5,inventory,"stockout_flag, ending_stock_units, inventory_v...",None,True
6,costs,"actual_unit_cost_eur, cost_variance_pct",None,True
7,returns,"return_units, return_rate_units",None,True
8,crm,"crm_activity_count, avg_sentiment_score",None,True
9,pipeline,"pipeline_opportunities, weighted_pipeline_eur",None,True


## 13.2 Define model features and build a column dictionary

In [32]:
MODEL_CATEGORICAL_FEATURES = [
    "product_line",
    "product_category",
    "region",
    "lifecycle_stage",
    "dominant_margin_category",
    "dominant_order_size_category",
    "dominant_rep_seniority",
    "campaign_channel",
]

MODEL_NUMERIC_FEATURES = [
    # Core sales and historical behavior
    "units_sold", "revenue", "unique_customers", "avg_discount_pct",
    "order_count", "active_sales_reps", "avg_asp_eur",
    "gross_profit_eur", "cogs_eur", "gross_margin_pct",
    "discount_value_eur", "outlier_order_count",
    "missing_sales_rep_count", "missing_discount_count",
    "missing_margin_count", "units_lag_1m", "units_lag_3m",
    "revenue_lag_1m", "rolling_units_mean_3m",
    "rolling_units_mean_6m", "rolling_units_std_3m",
    "volatility_3m", "mom_units_growth_pct",
    "customer_count_growth_pct", "trend_slope_3m", "trend_slope_6m",
    "risk_factor_sales_drop", "risk_factor_customer_drop",
    "risk_factor_volatility", "risk_factor_under_trend", "no_sales_flag",
    # Product, customer, region, and representative context
    "base_list_price_eur", "base_unit_cost_eur", "target_margin_pct",
    "product_growth_factor", "is_declining_product", "is_new_product",
    "product_age_months", "avg_customer_size_score",
    "avg_base_churn_probability", "enterprise_customer_count",
    "high_churn_customer_count", "covered_annual_quota_eur",
    "fx_to_eur", "market_growth_factor", "margin_factor",
    # Inventory, cost, and returns
    "opening_stock_units", "production_units", "ending_stock_units",
    "stockout_flag", "zero_stock_flag", "inventory_value_eur",
    "inventory_available_flag", "backorder_units",
    "supply_coverage_ratio", "standard_unit_cost_eur",
    "actual_unit_cost_eur", "cost_variance_eur", "cost_variance_pct",
    "return_units", "return_value_eur", "return_count",
    "return_rate_units",
    # CRM and pipeline
    "crm_activity_count", "crm_activity_minutes", "crm_demo_like_count",
    "avg_sentiment_score", "avg_customer_health_score",
    "pipeline_opportunities", "weighted_pipeline_eur",
    "expected_pipeline_eur", "avg_win_probability", "avg_days_to_close",
    "closed_won_count", "closed_lost_count", "pipeline_conversion_rate",
    # Market activity
    "campaign_flag", "campaign_spend_eur", "campaign_impressions",
    "campaign_clicks", "campaign_ctr_pct", "website_visits",
    "product_page_views", "demo_requests", "marketing_qualified_leads",
    "cost_per_lead_eur", "regional_crm_activity_index",
    "activity_pipeline_interest_index", "market_activity_available_flag",
    # Market signals
    "market_demand_index", "market_growth_pct",
    "competitor_pressure_index", "seasonality_index",
    "macro_business_index", "supply_pressure_index",
    "market_pipeline_interest_index", "demand_shock_flag",
    "market_opportunity_score", "regional_market_growth_factor",
    "market_signal_available_flag",
    # Calendar
    "Year", "Quarter", "Month", "YearMonthKey",
]

missing_model_features = [
    column
    for column in MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES
    if column not in enriched_modeling_dataset.columns
]
if missing_model_features:
    raise AssertionError(f"Missing model features: {missing_model_features}")

print(f"Numeric model features:     {len(MODEL_NUMERIC_FEATURES)}")
print(f"Categorical model features: {len(MODEL_CATEGORICAL_FEATURES)}")


Numeric model features:     104
Categorical model features: 8


In [33]:
base_definitions = {
    "product_id": "Unique product key.",
    "year_month": "Observation month in YYYY-MM format.",
    "product_line": "Product family mapped to the compatible EWS name.",
    "product_category": "Product group mapped to the compatible EWS name.",
    "region": "Geographical reporting region.",
    "units_sold": "Units sold in the product-region-month.",
    "revenue": "Revenue in EUR in the product-region-month.",
    "unique_customers": "Distinct purchasing customers.",
    "avg_discount_pct": "Mean transaction discount percentage.",
    "units_lag_1m": "Units sold one month earlier.",
    "units_lag_3m": "Units sold three months earlier.",
    "revenue_lag_1m": "Revenue one month earlier.",
    "rolling_units_mean_3m": "Mean units over the prior three months; current month excluded.",
    "rolling_units_mean_6m": "Mean units over the prior six months; current month excluded.",
    "rolling_units_std_3m": "Unit standard deviation over the prior three months.",
    "volatility_3m": "Prior-three-month standard deviation divided by prior-three-month mean.",
    "mom_units_growth_pct": "Current units versus previous-month units.",
    "customer_count_growth_pct": "Current unique customers versus previous month.",
    "trend_slope_3m": "Linear unit trend across the prior three months.",
    "trend_slope_6m": "Linear unit trend across the prior six months.",
    "stockout_flag": "Maximum inventory stockout flag for the month.",
    "backorder_units": "Units above opening stock plus production when inventory is available.",
    "market_demand_index": "Market demand index from market_signals; 100 is the baseline.",
    "competitor_pressure_index": "Competitor pressure index from market_signals.",
    "campaign_flag": "One when market_activity records an active campaign.",
    "website_visits": "Product-region website visits from market_activity.",
    "demo_requests": "Product-region demo requests from market_activity.",
    "risk_factor_sales_drop": "One when sales declined last 3 months.",
    "risk_factor_customer_drop": "One when customer-count growth is below -25%.",
    "risk_factor_volatility": "One when three-month volatility exceeds 75th quantile.",
    "risk_factor_under_trend": "One when units are below 75% of trend-adjusted expectation.",
    "next_month_risk_label": "Target: 0 low, 1 medium, 2 high based on next-month triggers.",
}

source_by_column = {}
for source_table, contributed_columns in SOURCE_FEATURE_MAP.items():
    for column in contributed_columns:
        source_by_column[column] = source_table
for column in [
    "campaign_flag", "campaign_channel", "campaign_spend_eur",
    "campaign_impressions", "campaign_clicks", "campaign_ctr_pct",
    "website_visits", "product_page_views", "demo_requests",
    "marketing_qualified_leads", "cost_per_lead_eur",
    "regional_crm_activity_index", "activity_pipeline_interest_index",
]:
    source_by_column[column] = "market_activity"
for column in [
    "market_demand_index", "market_growth_pct",
    "competitor_pressure_index", "seasonality_index",
    "macro_business_index", "supply_pressure_index",
    "market_pipeline_interest_index", "demand_shock_flag",
    "market_opportunity_score", "regional_market_growth_factor",
]:
    source_by_column[column] = "market_signals"

dictionary_rows = []
for column in enriched_modeling_dataset.columns:
    if column == TARGET_COL:
        role = "target"
    elif column in MODEL_NUMERIC_FEATURES:
        role = "numeric feature"
    elif column in MODEL_CATEGORICAL_FEATURES:
        role = "categorical feature"
    elif column in ["product_id", "region_id", "region", "year_month", "target_month"]:
        role = "identifier / time"
    else:
        role = "context / audit"

    dictionary_rows.append(
        {
            "column": column,
            "definition": base_definitions.get(
                column,
                column.replace("_", " ").capitalize() + ".",
            ),
            "source_table": source_by_column.get(column, "derived / joined"),
            "role": role,
        }
    )

ews_column_dictionary = pd.DataFrame(dictionary_rows)
display(ews_column_dictionary.head(40))


,column,definition,source_table,role
0,product_id,Unique product key.,derived / joined,identifier / time
1,year_month,Observation month in YYYY-MM format.,derived / joined,identifier / time
2,product_line,Product family mapped to the compatible EWS name.,products,categorical feature
3,product_category,Product group mapped to the compatible EWS name.,derived / joined,categorical feature
4,region,Geographical reporting region.,regions,categorical feature
5,units_sold,Units sold in the product-region-month.,sales,numeric feature
6,revenue,Revenue in EUR in the product-region-month.,sales,numeric feature
7,unique_customers,Distinct purchasing customers.,derived / joined,numeric feature
8,avg_discount_pct,Mean transaction discount percentage.,derived / joined,numeric feature
9,units_lag_1m,Units sold one month earlier.,derived / joined,numeric feature


# 14. Create chronological train, validation and test sets

Random splitting would let future market conditions influence training. We therefore split by `target_month`, the month whose risk is being predicted.

Preprocessing parameters are fitted on the training set only. Validation and test data use the already-fitted medians, means, standard deviations and category levels.


In [34]:
def create_time_based_splits(
    dataframe: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    labeled = dataframe.dropna(subset=[TARGET_COL]).copy()
    labeled[TARGET_COL] = labeled[TARGET_COL].astype(int)
    labeled["target_month"] = pd.to_datetime(labeled["target_month"])

    target_months = sorted(labeled["target_month"].unique())
    if len(target_months) < 12:
        raise ValueError(
            "At least 12 labeled target months are required for a robust split."
        )

    train_month_count = max(1, int(len(target_months) * 0.70))
    validation_month_count = max(1, int(len(target_months) * 0.15))

    train_months = target_months[:train_month_count]
    validation_months = target_months[
        train_month_count : train_month_count + validation_month_count
    ]
    test_months = target_months[
        train_month_count + validation_month_count :
    ]

    train = labeled[labeled["target_month"].isin(train_months)].copy()
    validation = labeled[
        labeled["target_month"].isin(validation_months)
    ].copy()
    test = labeled[labeled["target_month"].isin(test_months)].copy()

    split_summary = pd.DataFrame(
        [
            {
                "split": name,
                "rows": len(split),
                "observation_start": split["year_month"].min(),
                "observation_end": split["year_month"].max(),
                "target_start": split["target_month"].min(),
                "target_end": split["target_month"].max(),
            }
            for name, split in [
                ("train", train),
                ("validation", validation),
                ("test", test),
            ]
        ]
    )

    return train, validation, test, split_summary

train, validation, test, split_summary = create_time_based_splits(
    enriched_modeling_dataset
)
display(split_summary)

assert train["target_month"].max() < validation["target_month"].min()
assert validation["target_month"].max() < test["target_month"].min()
print("Chronological split checks passed.")


,split,rows,observation_start,observation_end,target_start,target_end
0,train,82000,2021-01,2024-05,2021-02-01,2024-06-01
1,validation,16000,2024-06,2025-01,2024-07-01,2025-02-01
2,test,20000,2025-02,2025-11,2025-03-01,2025-12-01


Chronological split checks passed.


## 14.1 Fit the preprocessing parameters on training data

The helper stores simple, auditable statistics:

- numeric missing values → training median;
- numeric scaling → training mean and standard deviation;
- categorical missing values → training mode;
- one-hot levels → categories observed in training.

The next cell displays part of the fitted state before transforming any data.

In [35]:
def fit_pandas_preprocessor(
    train_data: pd.DataFrame,
    numeric_features: list[str],
    categorical_features: list[str],
) -> dict[str, object]:
    numeric_medians = train_data[numeric_features].median(
        numeric_only=True
    ).fillna(0)
    imputed_numeric = train_data[numeric_features].fillna(numeric_medians)
    numeric_means = imputed_numeric.mean()
    numeric_stds = imputed_numeric.std().replace(0, 1).fillna(1)

    categorical_modes = {}
    categorical_levels = {}
    for column in categorical_features:
        mode = train_data[column].mode(dropna=True)
        categorical_modes[column] = (
            mode.iloc[0] if len(mode) else "Unknown"
        )
        values = (
            train_data[column]
            .fillna(categorical_modes[column])
            .astype(str)
        )
        categorical_levels[column] = sorted(values.unique().tolist())

    return {
        "numeric_features": numeric_features,
        "categorical_features": categorical_features,
        "numeric_medians": numeric_medians,
        "numeric_means": numeric_means,
        "numeric_stds": numeric_stds,
        "categorical_modes": categorical_modes,
        "categorical_levels": categorical_levels,
    }

preprocessor = fit_pandas_preprocessor(
    train,
    MODEL_NUMERIC_FEATURES,
    MODEL_CATEGORICAL_FEATURES,
)

fitted_numeric_example = pd.DataFrame(
    {
        "median": preprocessor["numeric_medians"],
        "mean": preprocessor["numeric_means"],
        "std": preprocessor["numeric_stds"],
    }
).head(15)
fitted_category_example = pd.DataFrame(
    {
        "feature": MODEL_CATEGORICAL_FEATURES,
        "training_mode": [
            preprocessor["categorical_modes"][column]
            for column in MODEL_CATEGORICAL_FEATURES
        ],
        "number_of_training_levels": [
            len(preprocessor["categorical_levels"][column])
            for column in MODEL_CATEGORICAL_FEATURES
        ],
    }
)

display(fitted_numeric_example)
display(fitted_category_example)


,median,mean,std
units_sold,11.000,37.631,64.497
revenue,"2,260.900","18,205.758","42,546.434"
unique_customers,1.000,0.940,1.094
avg_discount_pct,0.120,0.122,0.041
order_count,1.000,0.948,1.109
active_sales_reps,1.000,0.936,1.093
avg_asp_eur,456.330,482.706,308.573
gross_profit_eur,206.330,"4,147.988","11,203.175"
cogs_eur,"1,544.085","14,057.770","33,231.054"
gross_margin_pct,0.238,0.247,0.129


,feature,training_mode,number_of_training_levels
0,product_line,IT Devices,7
1,product_category,Accessory Kit,10
2,region,North America,8
3,lifecycle_stage,Mature,4
4,dominant_margin_category,Low Margin,3
5,dominant_order_size_category,Medium Order,3
6,dominant_rep_seniority,Professional,5
7,campaign_channel,No Active Campaign,7


In [36]:
def transform_with_pandas_preprocessor(
    dataframe: pd.DataFrame,
    fitted_preprocessor: dict[str, object],
) -> pd.DataFrame:
    numeric_features = fitted_preprocessor["numeric_features"]
    categorical_features = fitted_preprocessor["categorical_features"]

    numeric_data = dataframe[numeric_features].copy()
    numeric_data = numeric_data.fillna(
        fitted_preprocessor["numeric_medians"]
    )
    numeric_data = (
        numeric_data - fitted_preprocessor["numeric_means"]
    ) / fitted_preprocessor["numeric_stds"]

    encoded_parts = [numeric_data.reset_index(drop=True)]
    for column in categorical_features:
        values = (
            dataframe[column]
            .fillna(fitted_preprocessor["categorical_modes"][column])
            .astype(str)
        )
        levels = fitted_preprocessor["categorical_levels"][column]
        encoded = pd.DataFrame(
            {
                f"{column}__{level}": (values == level)
                .astype(int)
                .to_numpy()
                for level in levels
            }
        )
        encoded_parts.append(encoded)

    return pd.concat(encoded_parts, axis=1)

train_ml_ready = transform_with_pandas_preprocessor(train, preprocessor)
validation_ml_ready = transform_with_pandas_preprocessor(
    validation, preprocessor
)
test_ml_ready = transform_with_pandas_preprocessor(test, preprocessor)

feature_names = train_ml_ready.columns.tolist()
assert feature_names == validation_ml_ready.columns.tolist()
assert feature_names == test_ml_ready.columns.tolist()

train_ml_ready[TARGET_COL] = train[TARGET_COL].to_numpy()
validation_ml_ready[TARGET_COL] = validation[TARGET_COL].to_numpy()
test_ml_ready[TARGET_COL] = test[TARGET_COL].to_numpy()

transformed_summary = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "rows": [
            len(train_ml_ready),
            len(validation_ml_ready),
            len(test_ml_ready),
        ],
        "feature_columns": [
            len(feature_names), len(feature_names), len(feature_names)
        ],
        "missing_feature_cells": [
            int(train_ml_ready[feature_names].isna().sum().sum()),
            int(validation_ml_ready[feature_names].isna().sum().sum()),
            int(test_ml_ready[feature_names].isna().sum().sum()),
        ],
    }
)
display(transformed_summary)
display(train_ml_ready.head(3))


,split,rows,feature_columns,missing_feature_cells
0,train,82000,151,0
1,validation,16000,151,0
2,test,20000,151,0


,units_sold,revenue,unique_customers,avg_discount_pct,order_count,active_sales_reps,avg_asp_eur,gross_profit_eur,cogs_eur,gross_margin_pct,discount_value_eur,outlier_order_count,missing_sales_rep_count,missing_discount_count,missing_margin_count,units_lag_1m,units_lag_3m,revenue_lag_1m,rolling_units_mean_3m,rolling_units_mean_6m,rolling_units_std_3m,volatility_3m,mom_units_growth_pct,customer_count_growth_pct,trend_slope_3m,trend_slope_6m,risk_factor_sales_drop,risk_factor_customer_drop,risk_factor_volatility,risk_factor_under_trend,no_sales_flag,base_list_price_eur,base_unit_cost_eur,target_margin_pct,product_growth_factor,is_declining_product,is_new_product,product_age_months,avg_customer_size_score,avg_base_churn_probability,enterprise_customer_count,high_churn_customer_count,covered_annual_quota_eur,fx_to_eur,market_growth_factor,margin_factor,opening_stock_units,production_units,ending_stock_units,stockout_flag,...,Month,YearMonthKey,product_line__Accessories,product_line__Components,product_line__IT Devices,product_line__Industrial Tech,product_line__Legacy Products,product_line__Medical Devices,product_line__Services,product_category__Accessory Kit,product_category__Connectivity Module,product_category__Industrial Scanner,product_category__Laptop Pro,product_category__Laptop Standard,product_category__Legacy Workstation,product_category__Medical Sensor,product_category__Monitoring Device,product_category__Service Contract,product_category__Tablet Enterprise,region__APAC,region__Benelux,region__DACH,region__Eastern Europe,region__North America,region__Northern Europe,region__Southern Europe,region__Western Europe,lifecycle_stage__Decline,lifecycle_stage__Growth,lifecycle_stage__Mature,lifecycle_stage__New,dominant_margin_category__High Margin,dominant_margin_category__Low Margin,dominant_margin_category__Medium Margin,dominant_order_size_category__Large Order,dominant_order_size_category__Medium Order,dominant_order_size_category__Small Order,dominant_rep_seniority__Junior,dominant_rep_seniority__Key Account,dominant_rep_seniority__Professional,dominant_rep_seniority__Senior,dominant_rep_seniority__Unknown,campaign_channel__Content,campaign_channel__Email,campaign_channel__No Active Campaign,campaign_channel__Paid Search,campaign_channel__Partner,campaign_channel__Trade Fair,campaign_channel__Webinar,next_month_risk_label
0,0.672,0.897,0.968,-0.903,0.948,0.974,0.675,0.777,0.886,-0.144,0.302,-0.110,-0.076,-0.086,-0.062,-0.405,-0.391,-0.368,-0.246,-0.237,-0.190,-0.016,-0.155,-0.214,-0.005,-0.016,-0.129,-0.664,-0.533,-0.860,-0.875,0.609,0.568,-0.334,0.047,1.709,-0.264,-1.644,-0.247,-0.900,1.317,-0.330,0.721,0.549,0.223,0.095,-0.213,0.243,-0.287,-0.043,...,-1.463,-1.295,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0
1,-0.583,-0.428,-0.859,-0.044,-0.855,-0.856,-0.085,-0.370,-0.423,-0.063,-0.385,-0.110,-0.076,-0.086,-0.062,0.691,-0.391,0.917,-0.246,-0.237,-0.190,-0.016,-0.212,-0.944,-0.005,-0.016,-0.129,1.505,-0.533,-0.860,1.143,0.609,0.568,-0.334,0.047,1.709,-0.264,-1.581,-0.165,-0.105,-0.503,-0.330,-0.786,0.549,0.223,0.095,-0.199,-0.183,-0.197,-0.043,...,-1.175,-1.285,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0
2,-0.335,-0.157,0.055,-0.915,0.046,0.059,0.768,-0.105,-0.166,0.087,-0.247,-0.110,-0.076,-0.086,-0.062,-0.577,-0.391,-0.422,0.090,-0.237,0.413,0.465,-0.155,-0.214,-0.005,-0.016,-0.129,-0.664,-0.533,1.163,-0.875,0.609,0.568,-0.334,0.047,1.709,-0.264,-1.518,0.033,0.368,-0.503,-0.330,-0.140,0.549,0.223,0.095,-0.634,-0.718,-0.621,-0.043,...,-0.886,-1.275,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0,0,0,1,0,0


# 15. Final quality assurance

These checks protect the compatible schema, observation grain, target timing, binary flags, split chronology, transformed matrices, and use of all 13 sources.


In [37]:
binary_columns = [
    "risk_factor_sales_drop",
    "risk_factor_customer_drop",
    "risk_factor_volatility",
    "risk_factor_under_trend",
    "stockout_flag",
    "campaign_flag",
    "demand_shock_flag",
]

final_checks = {
    "compatible_32_column_schema": list(final_modeling_dataset.columns)
    == EWS_COLUMNS,
    "enriched_starts_with_compatible_schema": list(
        enriched_modeling_dataset.columns[: len(EWS_COLUMNS)]
    )
    == EWS_COLUMNS,
    "unique_product_region_month_key": not enriched_modeling_dataset.duplicated(
        ["product_id", "region_id", "year_month"]
    ).any(),
    "no_future_value_columns_exported": not any(
        column.startswith("future_")
        for column in enriched_modeling_dataset.columns
    ),
    "final_group_rows_are_unlabeled": enriched_modeling_dataset.groupby(
        ["product_id", "region_id"], dropna=False
    ).tail(1)[TARGET_COL].isna().all(),
    "risk_labels_are_valid": set(
        enriched_modeling_dataset[TARGET_COL].dropna().unique()
    ).issubset({0, 1, 2}),
    "binary_features_are_valid": all(
        set(enriched_modeling_dataset[column].dropna().unique()).issubset(
            {0, 1}
        )
        for column in binary_columns
    ),
    "all_sources_contribute": source_usage["used"].all(),
    "ml_matrices_have_no_missing_features": all(
        not dataframe[feature_names].isna().any().any()
        for dataframe in [
            train_ml_ready,
            validation_ml_ready,
            test_ml_ready,
        ]
    ),
    "split_order_is_chronological": (
        train["target_month"].max() < validation["target_month"].min()
        and validation["target_month"].max() < test["target_month"].min()
    ),
}

final_qa = pd.DataFrame(
    {"check": final_checks.keys(), "passed": final_checks.values()}
)
display(final_qa)

if not all(final_checks.values()):
    raise AssertionError("At least one final quality check failed.")

print("All final quality checks passed.")


,check,passed
0,compatible_32_column_schema,True
1,enriched_starts_with_compatible_schema,True
2,unique_product_region_month_key,True
3,no_future_value_columns_exported,True
4,final_group_rows_are_unlabeled,True
5,risk_labels_are_valid,True
6,binary_features_are_valid,True
7,all_sources_contribute,True
8,ml_matrices_have_no_missing_features,True
9,split_order_is_chronological,True


All final quality checks passed.


In [39]:
head = enriched_modeling_dataset.head(5000)

head.to_excel(
    OUTPUT_DIR_PREPROCESSED / "ews_modeling_dataset_enriched.xlsx", index=False
)

# 16. Export

In [40]:
# Analytical and feature-engineering outputs
final_modeling_dataset.to_csv(
    OUTPUT_DIR_PREPROCESSED / "ews_modeling_dataset.csv", index=False
)
enriched_modeling_dataset.to_csv(
    OUTPUT_DIR_PREPROCESSED / "ews_modeling_dataset_enriched.csv", index=False
)
customer_product_monthly.to_csv(
    OUTPUT_DIR_PREPROCESSED / "customer_product_monthly.csv", index=False
)
ews_column_dictionary.to_csv(
    OUTPUT_DIR_PREPROCESSED / "ews_column_dictionary.csv", index=False
)
rejected_sales.to_csv(
    OUTPUT_DIR_PREPROCESSED / "rejected_sales.csv", index=False
)

# ML-ready matrices
train_ml_ready.to_csv(
    OUTPUT_DIR_PREPROCESSED / "train_ml_ready.csv", index=False
)
validation_ml_ready.to_csv(
    OUTPUT_DIR_PREPROCESSED / "valid_ml_ready.csv", index=False
)
test_ml_ready.to_csv(
    OUTPUT_DIR_PREPROCESSED / "test_ml_ready.csv", index=False
)

id_export_columns = [
    "product_id", "year_month", "region", "region_id",
    "target_month", TARGET_COL,
]
train[id_export_columns].to_csv(
    OUTPUT_DIR_PREPROCESSED / "train_ids.csv", index=False
)
validation[id_export_columns].to_csv(
    OUTPUT_DIR_PREPROCESSED / "valid_ids.csv", index=False
)
test[id_export_columns].to_csv(
    OUTPUT_DIR_PREPROCESSED / "test_ids.csv", index=False
)


In [41]:
preprocessor_parameters = {
    "numeric_features": MODEL_NUMERIC_FEATURES,
    "categorical_features": MODEL_CATEGORICAL_FEATURES,
    "numeric_medians": preprocessor["numeric_medians"].to_dict(),
    "numeric_means": preprocessor["numeric_means"].to_dict(),
    "numeric_stds": preprocessor["numeric_stds"].to_dict(),
    "categorical_modes": preprocessor["categorical_modes"],
    "categorical_levels": preprocessor["categorical_levels"],
    "feature_names": feature_names,
}
with open(
    OUTPUT_DIR_PREPROCESSED / "preprocessor_parameters.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(preprocessor_parameters, file, indent=2, default=str)

quality_report = {
    "raw_shapes": {
        name: list(dataframe.shape) for name, dataframe in raw.items()
    },
    "canonical_sales_shape": list(sales.shape),
    "rejected_sales_rows": len(rejected_sales),
    "compatible_modeling_shape": list(final_modeling_dataset.shape),
    "enriched_modeling_shape": list(enriched_modeling_dataset.shape),
    "market_activity_coverage_pct": round(
        modeling["market_activity_available_flag"].mean() * 100, 4
    ),
    "market_signal_coverage_pct": round(
        modeling["market_signal_available_flag"].mean() * 100, 4
    ),
    "target_distribution": (
        final_modeling_dataset[TARGET_COL]
        .value_counts(dropna=False)
        .sort_index()
        .astype(int)
        .to_dict()
    ),
    "split_shapes": {
        "train": list(train_ml_ready.shape),
        "validation": list(validation_ml_ready.shape),
        "test": list(test_ml_ready.shape),
    },
    "ml_feature_count": len(feature_names),
    "final_checks": {key: bool(value) for key, value in final_checks.items()},
}
with open(
    OUTPUT_DIR_PREPROCESSED / "preprocessing_quality_report.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(quality_report, file, indent=2, default=str)


In [42]:
exported_files = []
for folder in [OUTPUT_DIR_PREPROCESSED]:
    for path in sorted(folder.glob("*")):
        if path.is_file():
            exported_files.append(
                {
                    "folder": folder.name,
                    "file": path.name,
                    "size_bytes": path.stat().st_size,
                }
            )

export_summary = pd.DataFrame(exported_files)
display(export_summary)
print(f"Export completed: {len(export_summary)} files created.")


,folder,file,size_bytes
0,preprocessed_data,customer_product_monthly.csv,10210426
1,preprocessed_data,ews_column_dictionary.csv,10118
2,preprocessed_data,ews_modeling_dataset.csv,27717679
3,preprocessed_data,ews_modeling_dataset_enriched.csv,100017485
4,preprocessed_data,ews_modeling_dataset_enriched.xlsx,3396166
5,preprocessed_data,preprocessing_quality_report.json,1743
6,preprocessed_data,preprocessor_parameters.json,22904
7,preprocessed_data,rejected_sales.csv,788
8,preprocessed_data,test_ids.csv,830075
9,preprocessed_data,test_ml_ready.csv,42746845


Export completed: 14 files created.
